In [ ]:
import pandas as pd
import time
import matplotlib.pyplot as plt
import numpy as np
import functions_changes
import datetime
import math
import re
import warnings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from IPython.display import display, HTML
from datetime import timedelta
import importlib
import json
from tqdm import tqdm
from itertools import islice
from itertools import combinations
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
warnings.filterwarnings('ignore')
np.random.seed(42)
from sqlalchemy import create_engine, text
import urllib
import os

# Provide an authorized SQL Server ODBC connection string through the environment.
_odbc_connection = os.environ.get("SEPSIS_ODBC_CONNECTION_STRING")
if not _odbc_connection:
    raise RuntimeError(
        "Set SEPSIS_ODBC_CONNECTION_STRING to access the restricted clinical database."
    )
_params = urllib.parse.quote_plus(_odbc_connection)
_engine = create_engine(f"mssql+pyodbc:///?odbc_connect={_params}")

def do_hhc_sql(sql_query):
    with _engine.connect() as conn:
        return pd.read_sql(text(sql_query), conn)

In [ ]:
%%time
# sql_query = f"""SELECT * FROM CAB_EPIC_Oxford.dbo.SepticFinalCohortUpdate"""
sql_query = f"SELECT * FROM {os.environ['SEPSIS_SEPTIC_COHORT_TABLE']}"
Sepsis_df = do_hhc_sql(sql_query)
patient_keys = Sepsis_df['PatientDurableKey'].drop_duplicates()
encounter_keys = Sepsis_df['EncounterKey'].drop_duplicates()
Sepsis_df['Label_sirs'] = 0
Sepsis_df = functions_changes.sepsis_dates(Sepsis_df, 'Label_sirs')
Sepsis_df = functions_changes.rename_columns(Sepsis_df)
Sepsis_df = functions_changes.apply_outlier_handling(Sepsis_df)
Sepsis_df = Sepsis_df.drop(columns=[col for col in Sepsis_df.columns if col.startswith('Dept_') or col == 'IsEdVisit'])
print(len(patient_keys))
print(len(encounter_keys))

In [ ]:
%%time
qur = functions_changes.execute_score(encounter_keys.to_list())
scores = do_hhc_sql(qur)

In [ ]:
%%time
# sql_query = f"""SELECT * FROM CAB_EPIC_Oxford.dbo.NonSepticFinalCohortUpdate"""
sql_query = f"SELECT * FROM {os.environ['SEPSIS_NONSEPTIC_COHORT_TABLE']}"
Non_Sepsis_df = do_hhc_sql(sql_query)
non_patient_keys = Non_Sepsis_df['PatientDurableKey'].drop_duplicates()
non_encounters_keys = Non_Sepsis_df['EncounterKey'].drop_duplicates()
Non_Sepsis_df['Label_sirs'] = 0
Non_Sepsis_df = functions_changes.sepsis_dates(Non_Sepsis_df, 'Label_sirs')
Non_Sepsis_df = functions_changes.rename_columns(Non_Sepsis_df)
Non_Sepsis_df = functions_changes.apply_outlier_handling(Non_Sepsis_df)
Non_Sepsis_df = Non_Sepsis_df.drop(columns=[col for col in Non_Sepsis_df.columns if col.startswith('Dept_') or col == 'IsEdVisit'])
print(len(non_patient_keys))
print(len(non_encounters_keys))

In [ ]:
%%time
def query_epic_scores(encounter_keys, batch_size=40000):
    encounter_key_batches = functions_changes.chunk_list(encounter_keys, batch_size)
    df_list = []

    for batch in encounter_key_batches:
        sql_query = functions_changes.execute_score(batch)
        df_batch = do_hhc_sql(sql_query)
        df_list.append(df_batch)

    return pd.concat(df_list, ignore_index=True)
enc_keys = non_encounters_keys.to_list()
scores_non = query_epic_scores(enc_keys)

In [ ]:
ValCohort = pd.read_excel("ValidationCohort_Final.xlsx")
ValCohort = ValCohort[['EncounterKey', 'True Label']]
ValCohort = ValCohort.rename(columns={"True Label":"Label"})
ValCohort['EncounterKey'] = ValCohort['EncounterKey'].astype(float)
val_non_encounters = ValCohort[ValCohort['Label'] == 'No']['EncounterKey'] 
val_non_encounters = set(val_non_encounters) 
val = Sepsis_df[Sepsis_df['EncounterKey'].isin(ValCohort['EncounterKey'])]
val.loc[val['EncounterKey'].isin(val_non_encounters), 'Label'] = 0
Sepsis_df = Sepsis_df[~Sepsis_df['EncounterKey'].isin(ValCohort['EncounterKey'])]

In [ ]:
DateArray = functions_changes.DateGenerator(2018, 2026)
DateArray = [int(x) for x in DateArray]
date_dict = {date: idx for idx, date in enumerate(DateArray)}

In [ ]:
%%time
Sepsis_df = Sepsis_df.sort_values(by=['Num', 'DateTimeKey'])
septic_dfs = [group.reset_index(drop=True) for _, group in Sepsis_df.groupby('Num')]
sizes = [df.shape[0] for df in septic_dfs]
filtered_patients_data = functions_changes.process_patient_data(Sepsis_df, sizes, date_dict, 6)
len(filtered_patients_data)

In [ ]:
%%time
Non_Sepsis_df = Non_Sepsis_df.sort_values(by=['Num', 'DateTimeKey'])
nonseptic_dfs = [group.reset_index(drop=True) for _, group in Non_Sepsis_df.groupby('Num')]
sizes_non = [df.shape[0] for df in nonseptic_dfs]
filtered_patients_data_non = functions_changes.process_patient_data(Non_Sepsis_df, sizes_non, date_dict, 6)
len(filtered_patients_data_non)

In [ ]:
%%time
val = val.sort_values(by=['Num', 'DateTimeKey'])
val_dfs = [group.reset_index(drop=True) for _, group in val.groupby('Num')]
sizes_val = [df.shape[0] for df in val_dfs]
filtered_patients_data_val = functions_changes.process_patient_data(val, sizes_val, date_dict, 6)
len(filtered_patients_data_val)

In [ ]:
json_data = []
with open('llama_results_final.jsonl', 'r') as file:
    for line in file:
        json_data.append(json.loads(line.strip()))

parsed_data = [(key, value) for item in json_data for key, value in item.items()]
rs2 = pd.DataFrame(parsed_data, columns=['EncounterKey', 'Llama_Label'])
rs2 = rs2.drop_duplicates()
rs2['Llama_Label'] = rs2['Llama_Label'].apply(lambda x: re.search(r'\b(Yes|No)\b', x).group(1) if re.search(r'\b(Yes|No)\b', x) else x)
rs2 = rs2[rs2['EncounterKey'] != 'No']
df_no_results = rs2[rs2['Llama_Label'] == 'No']
df_no_results['EncounterKey'] = df_no_results['EncounterKey'].astype(float)
df_no_results['Llama_Label'].value_counts()

In [ ]:
non_septic_dfs = []
non_septic_encs = []
new_filtered_patients_data = []
moved_dfs = []

for df in filtered_patients_data:
    encounter_key = df['EncounterKey'].iloc[0]
    if encounter_key in df_no_results['EncounterKey'].unique():
        df['Label_sirs'] = df['Label'].copy()  # <-- Copy original label values first
        df['Label'] = 0                         # <-- Then overwrite Label
        non_septic_dfs.append(df)
        non_septic_encs.append(encounter_key)

print(len(non_septic_dfs))
print(len(set(non_septic_encs)))

encounter_keys_to_move = set(non_septic_encs)  # for faster lookup
for df in filtered_patients_data:
    encounter_key = df['EncounterKey'].iloc[0]
    if encounter_key in encounter_keys_to_move:
        moved_dfs.append(df)
    else:
        new_filtered_patients_data.append(df)

filtered_patients_data = new_filtered_patients_data
filtered_patients_data_non.extend(moved_dfs)
filtered_data2 = filtered_patients_data + filtered_patients_data_non
print(len(filtered_data2))

In [ ]:
filtered_patients_data_cleaned = [
    df.drop(columns=functions_changes.get_drop_columns(df), errors='ignore') for df in filtered_patients_data
]
filtered_patients_data_non_cleaned = [
    df.drop(columns=functions_changes.get_drop_columns(df), errors='ignore') for df in filtered_patients_data_non
]
filtered_patients_data_val_cleaned = [
    df.drop(columns=functions_changes.get_drop_columns(df), errors='ignore') for df in filtered_patients_data_val
]
filtered_data = filtered_patients_data_cleaned + filtered_patients_data_non_cleaned
val_patients = {df['PatientDurableKey'].iloc[0] for df in filtered_patients_data_val}
filtered_data = [df for df in filtered_data if df['PatientDurableKey'].iloc[0] not in val_patients]
all_dataframes = filtered_data2 + filtered_patients_data_val
df_combined = pd.concat(all_dataframes, ignore_index=True, sort=False)

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, average_precision_score, auc
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from collections import defaultdict
import pandas as pd
import numpy as np
from tqdm import tqdm

# hours -> number of consecutive 8-hour segments (your window_size)
WINDOW_SIZES = {8: 1, 24: 3, 48: 6}
# WINDOW_SIZES = {24: 3, 48: 6}

# ---------------------------------------------------------------------------
# YOUR CELL 1 (split builder) -> function of window_size
# ---------------------------------------------------------------------------
def build_bootstrapped_splits_original(window_size, num_bootstrap_splits=5):
    patients_dict = defaultdict(list)
    for df in filtered_data:
        patient_id = df['PatientDurableKey'].iloc[0]
        patients_dict[patient_id].append(df)
 
    patients_with_encounters_in_list = set()
    for patient_id, dfs in patients_dict.items():
        for df in dfs:
            encounter_key = df['EncounterKey'].iloc[0]
            if encounter_key in non_septic_encs:
                patients_with_encounters_in_list.add(patient_id)
                break
 
    patients_with_encs = list(patients_with_encounters_in_list)
    other_patients = list(set(patients_dict.keys()) - set(patients_with_encs))
 
    bootstrapped_splits = []
    for i in range(num_bootstrap_splits):
        train_other_patients = resample(
            other_patients, replace=False,
            n_samples=int(0.7 * len(other_patients)), random_state=42 + i)
        test_patients = list(set(other_patients) - set(train_other_patients))
        train_patient_ids = patients_with_encs + train_other_patients
 
        # .copy() so the in-place drops below never mutate filtered_data /
        # filtered_patients_data_val_cleaned (critical when looping windows)
        patients_train = [df.copy() for pid in train_patient_ids for df in patients_dict[pid]]
        patients_test = [df.copy() for pid in test_patients for df in patients_dict[pid]]
 
        patients_train = functions_changes.drop_column_efficient(patients_train, 'Label_sirs')
        patients_test = functions_changes.drop_column_efficient(patients_test, 'Label_sirs')
        val_source = [df.copy() for df in filtered_patients_data_val_cleaned]
        patients_val = functions_changes.drop_column_efficient(val_source, 'Label_sirs')
 
        drop_cols = ['PatientDurableKey', 'Index', 'DurableKey', 'Culture Exam',
                     'Image Exam', 'Num', 'NaN_Percentage']
        for df in patients_train:
            df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
        for df in patients_test:
            df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
        for df in patients_val:
            df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
        # ----- new
        not_imputed = ['EncounterKey', 'Label', 'DateTimeKey']
        impute_cols = [c for c in patients_train[0].columns if c not in not_imputed]

        train_rows = pd.concat([df[impute_cols] for df in patients_train], ignore_index=True)
        fit_rows = train_rows.sample(n=min(200000, len(train_rows)), random_state=42 + i)

        imputer = IterativeImputer(max_iter=10, random_state=42 + i, keep_empty_features=True)
        imputer.fit(fit_rows)

        def impute_list(dfs):
            lengths = [len(df) for df in dfs]
            stacked = pd.concat([df[impute_cols] for df in dfs], ignore_index=True)
            filled = imputer.transform(stacked)
            start = 0
            for df, n in zip(dfs, lengths):
                df[impute_cols] = filled[start:start + n]
                start += n

        impute_list(patients_train)
        impute_list(patients_test)
        impute_list(patients_val)

        keys = ['EncounterKey', 'DateTimeKey']
        test_imputed = pd.concat([df[keys + impute_cols] for df in patients_test], ignore_index=True)
        val_imputed = pd.concat([df[keys + impute_cols] for df in patients_val], ignore_index=True)
        test_imputed[impute_cols] = test_imputed[impute_cols].astype('float32')   # saves memory
        val_imputed[impute_cols] = val_imputed[impute_cols].astype('float32')
        # ----- new
        feature_names = patients_train[0].columns.drop(['EncounterKey', 'Label']).tolist()
        X_train_all, y_train_all, train_encounter_keys, train_datetime_keys = [], [], [], []
        for patient_df in tqdm(patients_train, desc=f"[{window_size}] Train"):
            Xp, yp, ek, dk = functions_changes.create_sliding_windows(patient_df, window_size)
            X_train_all.append(Xp); y_train_all.append(yp)
            train_encounter_keys.extend(ek); train_datetime_keys.extend(dk)
 
        X_test_all, y_test_all, test_encounter_keys, test_datetime_keys = [], [], [], []
        for patient_df in tqdm(patients_test, desc=f"[{window_size}] Test"):
            Xp, yp, ek, dk = functions_changes.create_sliding_windows(patient_df, window_size)
            X_test_all.append(Xp); y_test_all.append(yp)
            test_encounter_keys.extend(ek); test_datetime_keys.extend(dk)
 
        X_val_all, y_val_all, val_encounter_keys, val_datetime_keys = [], [], [], []
        for patient_df in patients_val:
            Xp, yp, ek, dk = functions_changes.create_sliding_windows(patient_df, window_size)
            X_val_all.append(Xp); y_val_all.append(yp)
            val_encounter_keys.extend(ek); val_datetime_keys.extend(dk)
 
        X_train_combined = np.concatenate(X_train_all, axis=0)
        y_train_combined = np.concatenate(y_train_all, axis=0)
        X_test_combined = np.concatenate(X_test_all, axis=0)
        y_test_combined = np.concatenate(y_test_all, axis=0)
        X_val_combined = np.concatenate(X_val_all, axis=0)
        y_val_combined = np.concatenate(y_val_all, axis=0)
 
        scaler = StandardScaler()
        X_train_combined = scaler.fit_transform(X_train_combined)
        X_test_combined = scaler.transform(X_test_combined)
        X_val_combined = scaler.transform(X_val_combined)
 
        bootstrapped_splits.append({
            'X_train': X_train_combined, 'y_train': y_train_combined,
            'train_encounter_keys': train_encounter_keys,
            'train_datetime_keys': train_datetime_keys,
            'X_test': X_test_combined, 'y_test': y_test_combined,
            'test_encounter_keys': test_encounter_keys,
            'test_datetime_keys': test_datetime_keys,
            'X_val': X_val_combined, 'y_val': y_val_combined,
            'val_encounter_keys': val_encounter_keys,
            'val_datetime_keys': val_datetime_keys,
            'scaler': scaler,
            'feature_names': feature_names,
            
            'test_imputed': test_imputed,
            'val_imputed': val_imputed,
        })
    return bootstrapped_splits

CACHE_SPLITS = True       # False = recompute every split for every window (uses less memory)
USE_MICE = True           # False = native-NaN version (no imputation)

_split_setup = {}         # patient grouping, computed once per run
_split_cache = {}         # split number -> stacked, imputed data

_DROP_COLS = ['PatientDurableKey', 'Index', 'DurableKey', 'Culture Exam',
              'Image Exam', 'Num', 'NaN_Percentage']
_NOT_IMPUTED = ['EncounterKey', 'Label', 'DateTimeKey']


def _model_columns(df):
    """Columns a DataFrame has after the drops done in the original function."""
    return tuple(c for c in df.columns if c != 'Label_sirs' and c not in _DROP_COLS)


def _get_split_setup():
    if _split_setup:
        return _split_setup
    patients_dict = defaultdict(list)
    for df in filtered_data:
        patient_id = df['PatientDurableKey'].iloc[0]
        patients_dict[patient_id].append(df)

    llama_encs = set(non_septic_encs)
    patients_with_encounters_in_list = set()
    for patient_id, dfs in patients_dict.items():
        for df in dfs:
            encounter_key = df['EncounterKey'].iloc[0]
            if encounter_key in llama_encs:
                patients_with_encounters_in_list.add(patient_id)
                break

    patients_with_encs = list(patients_with_encounters_in_list)
    other_patients = list(set(patients_dict.keys()) - set(patients_with_encs))

    # The fast path needs every encounter to have the same columns in the same
    # order (true for this pipeline). If not, fall back to the original code.
    ref = None
    consistent = True
    for df in list(filtered_data) + list(filtered_patients_data_val_cleaned):
        cols = _model_columns(df)
        if ref is None:
            ref = cols
        elif cols != ref:
            consistent = False
            break

    _split_setup.update(patients_dict=patients_dict, patients_with_encs=patients_with_encs,
                        other_patients=other_patients, cols=list(ref), consistent=consistent)
    if not consistent:
        print("Encounters have different column layouts -> using the original (slow) split builder.")
    return _split_setup


def _prepare_split(i):
    if CACHE_SPLITS and i in _split_cache:
        return _split_cache[i]
    st = _get_split_setup()
    other_patients = st['other_patients']
    train_other_patients = resample(
        other_patients, replace=False,
        n_samples=int(0.7 * len(other_patients)), random_state=42 + i)
    test_patients = list(set(other_patients) - set(train_other_patients))
    train_patient_ids = st['patients_with_encs'] + train_other_patients

    pdict = st['patients_dict']
    sets = {'train': [df for pid in train_patient_ids for df in pdict[pid]],
            'test':  [df for pid in test_patients for df in pdict[pid]],
            'val':   list(filtered_patients_data_val_cleaned)}

    cols = st['cols']
    impute_cols = [c for c in cols if c not in _NOT_IMPUTED]
    feature_cols = [c for c in cols if c not in ('EncounterKey', 'Label', 'DateTimeKey')]

    big = {name: pd.concat(dfs, ignore_index=True)[cols] for name, dfs in sets.items()}
    lengths = {name: np.array([len(df) for df in dfs], dtype=np.int64) for name, dfs in sets.items()}

    test_imputed = val_imputed = None
    if USE_MICE:
        train_rows = big['train'][impute_cols]
        fit_rows = train_rows.sample(n=min(200000, len(train_rows)), random_state=42 + i)
        imputer = IterativeImputer(max_iter=10, random_state=42 + i, keep_empty_features=True)
        imputer.fit(fit_rows)
        for name in ('train', 'test', 'val'):
            big[name][impute_cols] = imputer.transform(big[name][impute_cols])
        keys = ['EncounterKey', 'DateTimeKey']
        test_imputed = big['test'][keys + impute_cols].copy()
        val_imputed = big['val'][keys + impute_cols].copy()
        test_imputed[impute_cols] = test_imputed[impute_cols].astype('float32')   # saves memory
        val_imputed[impute_cols] = val_imputed[impute_cols].astype('float32')

    data = {'feature_names': feature_cols, 'test_imputed': test_imputed, 'val_imputed': val_imputed}
    for name in ('train', 'test', 'val'):
        b = big[name]
        ln = lengths[name]
        bounds = np.concatenate([[0], np.cumsum(ln)])
        data[name] = {'F': b[feature_cols].to_numpy(dtype=np.float64),
                      'y': b['Label'].to_numpy(),
                      'enc': b['EncounterKey'].to_numpy(),
                      'dtk': b['DateTimeKey'].to_numpy(),
                      'lengths': ln,
                      # two encounters exactly as the original code saw them; used only
                      # to reproduce the original array memory layout (see _make_windows)
                      'sample_dfs': [b.iloc[bounds[j]:bounds[j + 1]].copy() for j in range(min(2, len(ln)))]}
    del big
    if CACHE_SPLITS:
        _split_cache[i] = data
    return data


def _make_windows(d, window_size):
    """Same output as calling functions_changes.create_sliding_windows on every
    encounter and concatenating, but for all encounters at once."""
    w = window_size
    lengths = d['lengths']
    starts = np.concatenate([[0], np.cumsum(lengths)[:-1]])
    nw = np.clip(lengths - w + 1, 0, None)
    ws = np.repeat(starts, nw) + (np.arange(nw.sum()) - np.repeat(np.cumsum(nw) - nw, nw))
    X = np.hstack([d['F'][ws + k] for k in range(w)])
    # Keep the same memory layout as the original (C- or Fortran-ordered): the
    # values are identical either way, but StandardScaler sums in a different
    # order for the two layouts, which changes the last decimal places.
    pieces = [functions_changes.create_sliding_windows(sdf, w)[0] for sdf in d['sample_dfs']]
    ref = np.concatenate(pieces, axis=0)
    if ref.flags['F_CONTIGUOUS'] and not ref.flags['C_CONTIGUOUS']:
        X = np.asfortranarray(X)
    else:
        X = np.ascontiguousarray(X)
    return X, d['y'][ws + w - 1], d['enc'][ws].tolist(), d['dtk'][ws + w - 1].tolist()


def build_bootstrapped_splits(window_size, num_bootstrap_splits=5):
    if not _get_split_setup()['consistent']:
        return build_bootstrapped_splits_original(window_size, num_bootstrap_splits)
    bootstrapped_splits = []
    for i in tqdm(range(num_bootstrap_splits), desc=f"[{window_size}] Splits"):
        data = _prepare_split(i)
        X_train_combined, y_train_combined, train_encounter_keys, train_datetime_keys = _make_windows(data['train'], window_size)
        X_test_combined, y_test_combined, test_encounter_keys, test_datetime_keys = _make_windows(data['test'], window_size)
        X_val_combined, y_val_combined, val_encounter_keys, val_datetime_keys = _make_windows(data['val'], window_size)

        scaler = StandardScaler()
        X_train_combined = scaler.fit_transform(X_train_combined)
        X_test_combined = scaler.transform(X_test_combined)
        X_val_combined = scaler.transform(X_val_combined)

        bootstrapped_splits.append({
            'X_train': X_train_combined, 'y_train': y_train_combined,
            'train_encounter_keys': train_encounter_keys,
            'train_datetime_keys': train_datetime_keys,
            'X_test': X_test_combined, 'y_test': y_test_combined,
            'test_encounter_keys': test_encounter_keys,
            'test_datetime_keys': test_datetime_keys,
            'X_val': X_val_combined, 'y_val': y_val_combined,
            'val_encounter_keys': val_encounter_keys,
            'val_datetime_keys': val_datetime_keys,
            'scaler': scaler,
            'feature_names': list(data['feature_names']),

            'test_imputed': data['test_imputed'],
            'val_imputed': data['val_imputed'],
        })
    return bootstrapped_splits

In [ ]:
non_septic_sirs = functions_changes.filter_dataframes_with_label_one(filtered_patients_data_non, 'Label_sirs')
non_septic_sirs_non = functions_changes.filter_dataframes_with_label_zero(filtered_patients_data_non, 'Label_sirs')
septic = pd.concat(filtered_patients_data, ignore_index=True)

sirs_pos_keys = {df['EncounterKey'].iloc[0] for df in non_septic_sirs}
sirs_neg_keys = {df['EncounterKey'].iloc[0] for df in non_septic_sirs_non}
overlap_keys = sirs_pos_keys & sirs_neg_keys
moved_dfs = [df for df in non_septic_sirs_non if df['EncounterKey'].iloc[0] in overlap_keys]
non_septic_sirs_final = non_septic_sirs + moved_dfs
non_septic_sirs_non_final = [df for df in non_septic_sirs_non if df['EncounterKey'].iloc[0] not in overlap_keys]

non_septic_sirs_pos = pd.concat(non_septic_sirs_final, ignore_index=True)
non_septic_sirs_neg = pd.concat(non_septic_sirs_non_final, ignore_index=True)

septic_encounters = set(septic['EncounterKey'].to_list())
sirs_pos_encounters = septic_encounters | set(non_septic_sirs_pos['EncounterKey'].to_list())
sirs_neg_encounters = septic_encounters | set(non_septic_sirs_neg['EncounterKey'].to_list())
print(len(septic['EncounterKey'].drop_duplicates()))
print(len(non_septic_sirs_pos['EncounterKey'].drop_duplicates()))
print(len(non_septic_sirs_neg['EncounterKey'].drop_duplicates()))

In [ ]:
derivation_encs = {df['EncounterKey'].iloc[0] for df in filtered_data}
before_encs = {df['EncounterKey'].iloc[0] for df in filtered_patients_data_cleaned + filtered_patients_data_non_cleaned}
removed = before_encs - derivation_encs

print("Removed (validation patients):", len(removed), "encounters |",
      len(removed & septic_encounters), "septic,", len(removed - septic_encounters), "non-septic")
print("Septic:          ", len(septic_encounters & derivation_encs))
print("Non-septic SIRS+:", len(set(non_septic_sirs_pos['EncounterKey']) & derivation_encs))
print("Non-septic SIRS-:", len(set(non_septic_sirs_neg['EncounterKey']) & derivation_encs))

In [ ]:
from sklearn.preprocessing import MinMaxScaler
squery = 'SELECT * FROM CAB_EPIC_Oxford.dbo.Medications'
Meds = do_hhc_sql(squery)
epic_total = pd.concat([scores, scores_non])
# Apply transformations
epic_total['TimeValue'] = epic_total['ScoreTimeOfDayKey'].apply(functions_changes.asignar_valor)
epic_total['ScoreDateKey'] = epic_total['ScoreDateKey'].astype(str)
epic_total['DateTimeKey'] = epic_total['ScoreDateKey'] + epic_total['TimeValue'].astype(str)
epic_total['ScoreDateKey'] = epic_total['ScoreDateKey'].astype('float64')
epic_total['DateTimeKey'] = epic_total['DateTimeKey'].astype('int64')
epic_total['EncounterKey'] = epic_total['EncounterKey'].astype('float64')
# Group and sort
epic_total = epic_total.groupby(['EncounterKey', 'DateTimeKey'], as_index=False)['TotalScore'].mean()
epic_total = epic_total.sort_values(by=['EncounterKey', 'DateTimeKey'])
epic_total = epic_total.rename(columns={'TotalScore': 'EpicScore'})
scaler = MinMaxScaler()
epic_total['NormalizedEpicScore'] = scaler.fit_transform(epic_total[['EpicScore']])

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

SOFA_ONLY = ['Mean FiO2', 'Mean PaO2', 'Mean Weight', 'Total Urine Output',
             'Mean Glasgow Coma Scale Score', 'Mean Lactic Acid', 'Mean Mean Arterial Pressure']
_sofa_cache = {}   # split number -> (test, val) SOFA-only inputs after imputation

def sofa_inputs_imputed(i):
    """Impute the 7 SOFA-only inputs for split i the same way as the model features:
    fit on training patients only, one imputer for both classes, then transform test/val.
    Predictors: the already-imputed model features (left unchanged) + the 7 SOFA columns."""
    if i in _sofa_cache:
        return _sofa_cache[i]
    d = _split_cache[i]
    keys = ['EncounterKey', 'DateTimeKey']
    feat = [c for c in d['test_imputed'].columns if c not in keys]      # the main imputer's columns

    raw = df_combined[keys + SOFA_ONLY].copy()
    raw['_ek'] = raw['EncounterKey'].astype('int64')
    raw['_dk'] = raw['DateTimeKey'].astype('int64')
    raw = raw.drop(columns=keys).drop_duplicates(['_ek', '_dk'])

    def add_sofa_cols(ek, dk, feats):
        out = feats.reset_index(drop=True).copy()
        out['_ek'] = np.asarray(ek).astype('int64')
        out['_dk'] = np.asarray(dk).astype('int64')
        return out.merge(raw, on=['_ek', '_dk'], how='left')[feat + SOFA_ONLY]

    # training rows only, same 200,000-row sample size as the main imputer
    tr = d['train']
    n = len(tr['enc'])
    idx = np.sort(np.random.RandomState(42 + i).choice(n, size=min(200000, n), replace=False))
    tr_feats = pd.DataFrame(tr['F'][idx], columns=d['feature_names'])[feat]
    fit_rows = add_sofa_cols(tr['enc'][idx], tr['dtk'][idx], tr_feats)

    imputer = IterativeImputer(max_iter=10, random_state=42 + i, keep_empty_features=True,
                               min_value=0)   # these 7 inputs can't be negative
    imputer.fit(fit_rows.to_numpy(dtype=np.float64))

    result = []
    for imp in (d['test_imputed'], d['val_imputed']):
        rows = add_sofa_cols(imp['EncounterKey'], imp['DateTimeKey'], imp[feat])
        filled = imputer.transform(rows.to_numpy(dtype=np.float64))[:, len(feat):]
        out = imp[keys].reset_index(drop=True).copy()
        out[SOFA_ONLY] = filled
        result.append(out)
    _sofa_cache[i] = tuple(result)
    return _sofa_cache[i]

In [ ]:
def overwrite_with_imputed(base, imp):
    """Replace the feature values in base with the imputed values, matched on encounter and time."""
    if imp is None:
        return base
    k = ['EncounterKey', 'DateTimeKey']
    cols = [c for c in imp.columns if c in base.columns and c not in k]
    b = base.drop(columns=cols).copy()
    b['_ek'] = b['EncounterKey'].astype('int64'); b['_dk'] = b['DateTimeKey'].astype('int64')
    i = imp[k + cols].drop_duplicates(k).copy()
    i['_ek'] = i['EncounterKey'].astype('int64'); i['_dk'] = i['DateTimeKey'].astype('int64')
    out = b.merge(i.drop(columns=k), on=['_ek', '_dk'], how='left').drop(columns=['_ek', '_dk'])
    out[cols] = out[cols].fillna(base[cols].reset_index(drop=True))   # safety: keep raw if no match
    return out[base.columns]
    
_rule_cache = {}   # split number -> per-row SIRS/SOFA/Epic values (FAST)
 
def evaluate_and_build_curve_data(bootstrapped_splits, encounter_df_map, subgroups):
    def new_metric_block():
        return {'test': {'overall': [], 'septic': [], 'non_septic_sirs_pos': [], 'non_septic_sirs_neg': []},
                'val': {'overall': [], 'septic': [], 'non_septic_sirs_pos': [], 'non_septic_sirs_neg': []}}
 
    overall_aucs, val_aucs = [], []
    subgroup_aucs = {'septic': [], 'non_septic_sirs_pos': [], 'non_septic_sirs_neg': []}
    rule_based_metrics = {'sirs': new_metric_block(), 'sofa': new_metric_block(), 'epic': new_metric_block()}
 
    curve_data = {g: {m: {'y_true': [], 'y_score': []}
                      for m in ['model', 'sirs', 'sofa', 'epic']} for g in ['test', 'val']}
 
    def process_rule_based(scores_dict, test_scores, val_scores, score_name):
        scores_dict['test']['overall'].append(roc_auc_score(test_scores['Label'], test_scores[score_name]))
        scores_dict['val']['overall'].append(roc_auc_score(val_scores['Label'], val_scores[score_name]))
        for group, key_set in subgroups:
            sub = test_scores[test_scores['EncounterKey'].isin(key_set)]
            scores_dict['test'][group].append(roc_auc_score(sub['Label'], sub[score_name]))
 
    for i, split in enumerate(tqdm(bootstrapped_splits, desc="Bootstraps")):
        X_train, y_train = split['X_train'], split['y_train']
        X_test, y_test = split['X_test'], split['y_test']
        X_val, y_val = split['X_val'], split['y_val']
        test_keys, val_keys = split['test_encounter_keys'], split['val_encounter_keys']
        test_times, val_times = split['test_datetime_keys'], split['val_datetime_keys']
 
        model = XGBClassifier(n_estimators=100, max_depth=12, random_state=0, n_jobs=64)
        model.fit(X_train, y_train)
        if i == 0:                                 
            shap_inputs = {'model': model, 'X_test': X_test}
            
        y_test_prob = model.predict_proba(X_test)[:, 1]
        y_val_prob = model.predict_proba(X_val)[:, 1]
 
        overall_aucs.append(roc_auc_score(y_test, y_test_prob))
        val_aucs.append(roc_auc_score(y_val, y_val_prob))
        for group, key_set in subgroups:
            idx = [j for j, key in enumerate(test_keys) if key in key_set]
            subgroup_aucs[group].append(roc_auc_score(y_test[idx], y_test_prob[idx]))
 
        df_windows = pd.DataFrame({'EncounterKey': test_keys, 'DateTimeKey': test_times, 'Label': y_test})
        df_val_windows = pd.DataFrame({'EncounterKey': val_keys, 'DateTimeKey': val_times, 'Label': y_val})
 
        # FAST: the per-row SIRS / SOFA / Epic values of split i depend only on
        # which patients are in split i and on its imputation, which are the same
        # for the 8h, 24h and 48h windows -> computed once per split and reused.
        if CACHE_SPLITS and i in _rule_cache:
            (sirs_test_rows, sirs_val_rows, sofa_test_rows, sofa_val_rows,
             epic_test_rows, epic_val_rows) = _rule_cache[i]
        else:
            test_encounter_dfs = [pd.concat(encounter_df_map[key], ignore_index=True) for key in set(test_keys)]
            test_df = pd.concat(test_encounter_dfs, ignore_index=True)
            test_df = test_df.drop(columns=[c for c in test_df.columns if c == 'Label_sirs'])
            val_df = pd.concat(filtered_patients_data_val, ignore_index=True)
 
            test_df = overwrite_with_imputed(test_df, split.get('test_imputed'))
            val_df = overwrite_with_imputed(val_df, split.get('val_imputed'))
 
            sirs_test_rows = functions_changes.meet_sirs(test_df)[['EncounterKey', 'DateTimeKey', 'Sirs']]
            sirs_val_rows = functions_changes.meet_sirs(val_df)[['EncounterKey', 'DateTimeKey', 'Sirs']]
            # sofa_test_rows = functions_changes.process_population_sofa(test_df, Meds, df_combined)[['EncounterKey', 'DateTimeKey', 'NormalizedSofaScore']]
            # sofa_val_rows = functions_changes.process_population_sofa(val_df, Meds)[['EncounterKey', 'DateTimeKey', 'NormalizedSofaScore']]
            test_sofa, val_sofa = sofa_inputs_imputed(i)
            sofa_test_rows = functions_changes.process_population_sofa(test_df, Meds, test_sofa)[['EncounterKey', 'DateTimeKey', 'NormalizedSofaScore']]
            sofa_val_rows = functions_changes.process_population_sofa(overwrite_with_imputed(val_df, val_sofa), Meds)[['EncounterKey', 'DateTimeKey', 'NormalizedSofaScore']]
            epic_test_rows = functions_changes.process_epic_scores(epic_total, test_df)[['EncounterKey', 'DateTimeKey', 'NormalizedEpicScore']]
            epic_val_rows = functions_changes.process_epic_scores(epic_total, val_df)[['EncounterKey', 'DateTimeKey', 'NormalizedEpicScore']]
            if CACHE_SPLITS:
                _rule_cache[i] = (sirs_test_rows, sirs_val_rows, sofa_test_rows, sofa_val_rows,
                                  epic_test_rows, epic_val_rows)
 
        # --- SIRS ---
        sirs_test_scores = df_windows.merge(sirs_test_rows, on=['EncounterKey', 'DateTimeKey'], how='left').dropna()
        sirs_val_scores = df_val_windows.merge(sirs_val_rows, on=['EncounterKey', 'DateTimeKey'], how='left').dropna()
        # --- SOFA ---
        sofa_test_scores = df_windows.merge(sofa_test_rows, on=['EncounterKey', 'DateTimeKey'], how='left').dropna()
        sofa_val_scores = df_val_windows.merge(sofa_val_rows, on=['EncounterKey', 'DateTimeKey'], how='left').dropna()
        # --- EPIC ---
        epic_test_scores = df_windows.merge(epic_test_rows, on=['EncounterKey', 'DateTimeKey'], how='left').dropna()
        epic_val_scores = df_val_windows.merge(epic_val_rows, on=['EncounterKey', 'DateTimeKey'], how='left').dropna()
 
        process_rule_based(rule_based_metrics['sirs'], sirs_test_scores, sirs_val_scores, 'Sirs')
        process_rule_based(rule_based_metrics['sofa'], sofa_test_scores, sofa_val_scores, 'NormalizedSofaScore')
        process_rule_based(rule_based_metrics['epic'], epic_test_scores, epic_val_scores, 'NormalizedEpicScore')
 
        curve_data['test']['model']['y_true'].append(np.array(y_test))
        curve_data['test']['model']['y_score'].append(np.array(y_test_prob))
        curve_data['val']['model']['y_true'].append(np.array(y_val))
        curve_data['val']['model']['y_score'].append(np.array(y_val_prob))
        curve_data['test']['sirs']['y_true'].append(sirs_test_scores['Label'].values)
        curve_data['test']['sirs']['y_score'].append(sirs_test_scores['Sirs'].values)
        curve_data['val']['sirs']['y_true'].append(sirs_val_scores['Label'].values)
        curve_data['val']['sirs']['y_score'].append(sirs_val_scores['Sirs'].values)
        curve_data['test']['sofa']['y_true'].append(sofa_test_scores['Label'].values)
        curve_data['test']['sofa']['y_score'].append(sofa_test_scores['NormalizedSofaScore'].values)
        curve_data['val']['sofa']['y_true'].append(sofa_val_scores['Label'].values)
        curve_data['val']['sofa']['y_score'].append(sofa_val_scores['NormalizedSofaScore'].values)
        curve_data['test']['epic']['y_true'].append(epic_test_scores['Label'].values)
        curve_data['test']['epic']['y_score'].append(epic_test_scores['NormalizedEpicScore'].values)
        curve_data['val']['epic']['y_true'].append(epic_val_scores['Label'].values)
        curve_data['val']['epic']['y_score'].append(epic_val_scores['NormalizedEpicScore'].values)
 
    results = {'overall_aucs': overall_aucs, 'val_aucs': val_aucs,
               'subgroup_aucs': subgroup_aucs, 'rule_based_metrics': rule_based_metrics}
    return curve_data, results, shap_inputs
 
def summarize(name, scores):
    clean = [s for s in scores if not np.isnan(s)]
    if not clean:
        print(f"{name}: No valid scores"); return
    print(f"{name} AUC: {np.mean(clean):.4f} ± {np.std(clean, ddof=1):.4f} (mean ± SD, {len(clean)} splits)")

In [ ]:
def run_all_windows():
    _split_setup.clear(); _split_cache.clear(); _rule_cache.clear(); _sofa_cache.clear()
    # window-independent objects, built once
    encounter_df_map = defaultdict(list)
    for df in filtered_data:
        encounter_df_map[df['EncounterKey'].iloc[0]].append(df)
    subgroups = [('septic', septic_encounters),
                 ('non_septic_sirs_pos', sirs_pos_encounters),
                 ('non_septic_sirs_neg', sirs_neg_encounters)]
 
    all_curve_data, all_results, all_shap_inputs = {}, {}, {}
    for hours, wsize in WINDOW_SIZES.items():
        print(f"\n{'='*60}\n{hours}h window (window_size={wsize})\n{'='*60}")
        splits = build_bootstrapped_splits(wsize)
        feature_names_per_interval = splits[0]['feature_names'] 
        cd, res, shap_in = evaluate_and_build_curve_data(splits, encounter_df_map, subgroups)
        all_curve_data[hours] = cd
        all_results[hours] = res
        shap_in['window_size'] = wsize                                          # <-- ADD
        shap_in['feature_names'] = feature_names_per_interval                   # <-- ADD (see note below)
        all_shap_inputs[hours] = shap_in    
 
        print(f"\n🔹 [{hours}h] Overall AUCs:")
        summarize("Model (Test)", res['overall_aucs'])
        summarize("Model (Val)", res['val_aucs'])
 
        print(f"\n🔹 [{hours}h] Subgroup AUCs:")
        for group, scores in res['subgroup_aucs'].items():
            summarize(f"Model ({group})", scores)
 
        print(f"\n🔹 [{hours}h] Rule-Based Scores:")
        for score in ['sirs', 'sofa', 'epic']:
            print(f"\n--- {score.upper()} ---")
            for dataset in ['test', 'val']:
                print(f"  {dataset.capitalize()}:")
                summarize(f"{score} (Overall)", res['rule_based_metrics'][score][dataset]['overall'])
                if dataset == 'test':   # subgroups are computed for test only
                    for group in ['septic', 'non_septic_sirs_pos', 'non_septic_sirs_neg']:
                        summarize(f"{score} ({group})", res['rule_based_metrics'][score][dataset][group])
 
    return all_curve_data, all_results, all_shap_inputs

In [ ]:
all_curve_data, all_results, all_shap_inputs = run_all_windows()

In [ ]:
import pickle
with open('results_final_run.pkl', 'wb') as f:
    pickle.dump({'all_results': all_results, 'all_curve_data': all_curve_data}, f)
print("saved")

In [ ]:
for hours, res in all_results.items():
    print(f"\n{'='*60}\n{hours}h window\n{'='*60}")
    summarize("Model (Test)", res['overall_aucs'])
    summarize("Model (Val)", res['val_aucs'])
    for group, scores in res['subgroup_aucs'].items():
        summarize(f"Model ({group})", scores)
    for score in ['sirs', 'sofa', 'epic']:
        print(f"\n--- {score.upper()} ---")
        for dataset in ['test', 'val']:
            summarize(f"{score} {dataset} (Overall)", res['rule_based_metrics'][score][dataset]['overall'])
            if dataset == 'test':
                for group in ['septic', 'non_septic_sirs_pos', 'non_septic_sirs_neg']:
                    summarize(f"{score} ({group})", res['rule_based_metrics'][score][dataset][group])

In [ ]:
"""
All Figure-3-family plots in one module (npj-ready).

Main text (one window, 3 x 2):
    plot_main_fig(curve_data_8, window=8)        -> ROC / calibration / PR, test|val

Supplementary (two windows stacked as rows, 2 x 2 each):
    cds = {24: curve_data_24, 48: curve_data_48}
    plot_supp_roc(cds)      -> S1
    plot_supp_calib(cds)    -> S2
    plot_supp_pr(cds)       -> S3

Each `curve_data` is the dict your modeling loop builds:
    curve_data[group][method]['y_true' | 'y_score'],  group in {'test','val'},
    method in {'model','sirs','sofa','epic'} (lists, one entry per bootstrap fold).

SIRS in PR panels is drawn as a single operating-point marker (a fixed rule has
one operating point, not a tunable curve); set sirs_as_point=False to revert.
Set sirs_threshold=2 if the SIRS score is a 0-4 criteria count rather than 0/1.
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score
from sklearn.calibration import calibration_curve
from matplotlib.lines import Line2D
import matplotlib as mpl
import matplotlib.font_manager as fm
from matplotlib.font_manager import findfont, FontProperties

FONT_DIR = os.environ.get('SEPSIS_FONT_DIR', '')
for f in ['arial.ttf', 'arialbd.ttf', 'ariali.ttf', 'arialbi.ttf']:
    if FONT_DIR and os.path.isfile(f'{FONT_DIR}/{f}'):
        fm.fontManager.addfont(f'{FONT_DIR}/{f}')
# --- Nature / npj figure rcParams -----------------------------------------
# plt.rcParams.update({
#     'font.sans-serif': ['Arial'],
#     'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
#     'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
#     'figure.dpi': 350, 'savefig.dpi': 350, 'axes.linewidth': 0.6,
#     # 'font.family': 'sans-serif',
#     'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
# })
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 7, 'axes.labelsize':7, 'axes.titlesize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 6,
    # 'figure.figsize': (6.69, 7.5),
    'axes.linewidth': 1.0, 'lines.linewidth': 1.5,
    'xtick.major.width': 1.0, 'ytick.major.width': 1.0,
    'axes.prop_cycle': mpl.cycler(color=['#0072B2','#E69F00','#009E73','#CC79A7']),
    'figure.dpi': 350, 'savefig.dpi': 350,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
})

for w, s in [('normal','normal'), ('bold','normal'),
             ('normal','italic'), ('bold','italic')]:
    print(w, s, '->', findfont(FontProperties(family='Arial', weight=w, style=s)))
    
COLORS = {'model': '#2980b9', 'sirs': '#e74c3c', 'sofa': '#f39c12', 'epic': '#27ae60'}
STYLES = {'model': '-', 'sirs': '--', 'sofa': '-.', 'epic': ':'}
COHORT = {'test': 'testing set of derivation', 'val': 'validation'}


def _label(mth, window):
    return f'STRIDE-{window}' if mth == 'model' else {
        'sirs': 'SIRS', 'sofa': 'SOFA', 'epic': 'Epic'}[mth]


# --- curve builders --------------------------------------------------------

def _roc(y_trues, y_scores, n=100):
    mean_fpr = np.linspace(0, 1, n)
    tprs = []
    for y_true, y_score in zip(y_trues, y_scores):
        fpr, tpr, _ = roc_curve(y_true, y_score)
        it = np.interp(mean_fpr, fpr, tpr); it[0] = 0.0
        tprs.append(it)
    tprs = np.array(tprs); m = tprs.mean(0); s = tprs.std(0)
    return mean_fpr, m, np.maximum(m - 1.96 * s, 0), np.minimum(m + 1.96 * s, 1)


def _calib(y_trues, y_scores, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    curves = []
    for y_true, y_score in zip(y_trues, y_scores):
        try:
            pt, pp = calibration_curve(y_true, y_score, n_bins=n_bins, strategy='uniform')
            curves.append(np.interp(centers, pp, pt, left=np.nan, right=np.nan))
        except Exception:
            continue
    curves = np.array(curves)
    return (centers, np.nanmean(curves, 0),
            np.nanpercentile(curves, 2.5, 0), np.nanpercentile(curves, 97.5, 0))


def _pr(y_trues, y_scores, n=100):
    rr = np.linspace(0, 1, n)
    curves = []
    for y_true, y_score in zip(y_trues, y_scores):
        try:
            precision, recall, _ = precision_recall_curve(y_true, y_score)
            curves.append(np.interp(rr, recall[::-1], precision[::-1], left=np.nan, right=np.nan))
        except Exception:
            continue
    curves = np.array(curves)
    return (rr, np.nanmean(curves, 0),
            np.nanpercentile(curves, 2.5, 0), np.nanpercentile(curves, 97.5, 0))


def _sirs_point(y_trues, y_scores, threshold=1):
    recs, precs = [], []
    for y_true, y_score in zip(y_trues, y_scores):
        y_true = np.asarray(y_true); pred = np.asarray(y_score) >= threshold
        tp = np.sum(pred & (y_true == 1)); fp = np.sum(pred & (y_true == 0))
        fn = np.sum(~pred & (y_true == 1))
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
    recs, precs = np.array(recs), np.array(precs)
    return (np.nanmean(recs), np.nanmean(precs),
            np.nanpercentile(recs, 2.5), np.nanpercentile(recs, 97.5),
            np.nanpercentile(precs, 2.5), np.nanpercentile(precs, 97.5))


def _band(ax, x, m, lo, hi, key, window):
    ax.plot(x, m, label=_label(key, window), color=COLORS[key],
            linestyle=STYLES[key], linewidth=1.2)
    ax.fill_between(x, lo, hi, color=COLORS[key], alpha=0.2)


# --- single-panel drawers --------------------------------------------------

def _panel_roc(ax, cd_group, window, wtxt):
    for mth in ['model', 'sirs', 'sofa', 'epic']:
        d = cd_group[mth]
        _band(ax, *_roc(d['y_true'], d['y_score']), mth, window)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.3)
    ax.set_xlabel('False positive rate\n(1 - specificity)')
    ax.legend(loc='lower right', frameon=True)
    ax.set_title(f'ROC curve for {wtxt}')
    ax.grid(False)


def _panel_calib(ax, cd_group, window, wtxt):
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.3)
    for mth in ['model', 'sofa', 'epic']:
        d = cd_group[mth]
        _band(ax, *_calib(d['y_true'], d['y_score']), mth, window)
    ax.set_xlabel('Predicted probability')
    ax.legend(loc='upper left', frameon=True)
    ax.set_title(f'Calibration curve for {wtxt}')
    ax.grid(False)


def _panel_pr(ax, cd_group, window, wtxt, gtag, sirs_as_point, sirs_threshold):
    for mth in ['model', 'sofa', 'epic']:
        d = cd_group[mth]
        _band(ax, *_pr(d['y_true'], d['y_score']), mth, window)
        ap = np.mean([average_precision_score(yt, ys)
                      for yt, ys in zip(d['y_true'], d['y_score'])])
        print(f"{gtag:20} {_label(mth, window):9} AP: {ap:.4f}")
    d = cd_group['sirs']
    if sirs_as_point:
        r, p, rlo, rhi, plo, phi = _sirs_point(d['y_true'], d['y_score'], sirs_threshold)
        xerr = [[max(r - rlo, 0.0)], [max(rhi - r, 0.0)]]
        yerr = [[max(p - plo, 0.0)], [max(phi - p, 0.0)]]
        ax.errorbar(r, p, xerr=xerr, yerr=yerr,
            fmt='o', color=COLORS['sirs'], markersize=4, capsize=2,
            elinewidth=0.8, label='_nolegend_')
        sirs_handle = Line2D([0], [0], marker='o', linestyle='None',
                         color=COLORS['sirs'], markersize=4,
                         label='SIRS')
    else:
        _band(ax, *_pr(d['y_true'], d['y_score']), 'sirs', window)
    ax.set_xlabel('Recall')
    handles, _ = ax.get_legend_handles_labels()
    if sirs_handle is not None:
        handles.append(sirs_handle)
    ax.legend(handles=handles, loc='upper right', frameon=True)
    ax.set_title(f'Precision-recall curve for {wtxt}')
    ax.grid(False)


def _letters(axs):
    flat = axs.ravel()
    for ax, letter in zip(flat, 'abcdefgh'):
        ax.text(-0.18, 1.12, letter, transform=ax.transAxes,
                fontsize=8, fontweight='bold', va='top')


def _save(fig, outfile, outdir):
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, outfile)
    # fig.tight_layout()
    fig.savefig(path)#, bbox_inches='tight')
    fig.savefig(path.replace('.pdf', '.png'), dpi=350)
    plt.show()


# --- MAIN TEXT: one window, 3 x 2 -----------------------------------------

def plot_main_fig(curve_data, window=8, outfile=None, outdir='figures',
                  sirs_as_point=True, sirs_threshold=1):
    outfile = outfile or f'fig3_{window}h.pdf'
    #fig, axs = plt.subplots(3, 2, figsize=(7.09, 9.2), sharey='row')
    fig, axs = plt.subplots(3, 2, figsize=(5.31, 6.69), layout='constrained')
    for j, group in enumerate(['test', 'val']):
        wtxt = f'{window} hour time window\n({COHORT[group]} cohort)'
        _panel_roc(axs[0, j], curve_data[group], window, wtxt)
        _panel_calib(axs[1, j], curve_data[group], window, wtxt)
        _panel_pr(axs[2, j], curve_data[group], window, wtxt,
                  f'{window}h {group.upper()}', sirs_as_point, sirs_threshold)
        if j == 0:
            axs[0, j].set_ylabel('True positive rate\n(sensitivity)')
            axs[1, j].set_ylabel('Observed frequency')
            axs[2, j].set_ylabel('Precision')
    for row, letter in zip(range(3), ['a', 'b', 'c']):
        axs[row, 0].text(-0.18, 1.12, letter, transform=axs[row, 0].transAxes,
                         fontsize=12, fontweight='bold', va='top')
    _save(fig, outfile, outdir)
    return fig


# --- SUPPLEMENTARY: two windows as rows, 2 x 2 -----------------------------

def _supp(kind, cds, outfile, outdir, windows=(24, 48),
          sirs_as_point=True, sirs_threshold=1):
    # fig, axs = plt.subplots(2, 2, figsize=(7.09, 7.2), sharey=True)
    fig, axs = plt.subplots(2, 2, figsize=(5.31, 4.46), layout='constrained')
    for i, window in enumerate(windows):
        cd = cds[window]
        for j, group in enumerate(['test', 'val']):
            ax = axs[i, j]
            wtxt = f'{window} hour time window\n({COHORT[group]} cohort)'
            if kind == 'roc':
                _panel_roc(ax, cd[group], window, wtxt)
                ylab = 'True positive rate\n(sensitivity)'
            elif kind == 'calib':
                _panel_calib(ax, cd[group], window, wtxt)
                ylab = 'Observed frequency'
            else:
                _panel_pr(ax, cd[group], window, wtxt,
                          f'{window}h {group.upper()}', sirs_as_point, sirs_threshold)
                ylab = 'Precision'
            if j == 0:
                ax.set_ylabel(ylab)
    _letters(axs)
    _save(fig, outfile, outdir)
    return fig


def plot_supp_roc(cds, outfile='figS1_roc_24_48.pdf', outdir='figures', **kw):
    return _supp('roc', cds, outfile, outdir, **kw)


def plot_supp_calib(cds, outfile='figS2_calib_24_48.pdf', outdir='figures', **kw):
    return _supp('calib', cds, outfile, outdir, **kw)


def plot_supp_pr(cds, outfile='figS3_pr_24_48.pdf', outdir='figures', **kw):
    return _supp('pr', cds, outfile, outdir, **kw)


cds = {24: all_curve_data[24], 48: all_curve_data[48]}
# plot_supp_roc(cds)
# plot_supp_calib(cds)
# plot_supp_pr(cds)

In [ ]:
plot_main_fig(all_curve_data[8])

In [ ]:
plot_supp_roc(cds, sirs_as_point=True)

In [ ]:
plot_supp_calib(cds)

In [ ]:
plot_supp_pr(cds)

In [ ]:
"""
SHAP feature-importance figure for STRIDE-8/24/48 (Fig. 5).

Produces ONE combined PDF in the a/b-top, c-centered-below layout:

        [   a   ] [   b   ]
             [   c   ]

Requires, per window, a dict:
    'model'         : fitted XGBClassifier (fold 0)
    'X_test'        : that fold's test matrix
    'window_size'   : 1, 3, or 6   (inferred if omitted)
    'feature_names' : PER-INTERVAL feature names (len = n_features)

Usage:
    from shap_panels import shap_figure
    shap_figure(all_shap_inputs, outfile='fig5_shap.pdf')     # a=8h, b=24h, c=48h
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 7, 'axes.labelsize': 7, 'axes.titlesize': 7,
    'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
})

_CMAP = mpl.colors.LinearSegmentedColormap.from_list("custom_cmap", ["#a6dec1", "#c78a3d"])
# _CMAP = plt.cm.cividis

_STAT = {"Maximum": "Max", "Minimum": "Min", "Mean": "Mean"}
_VAR = {
    "Pulse": "HR", "Respiratory Rate": "RR", "Temperature": "Temp",
    "White Blood Cell Count": "WBC", "Systolic Blood Pressure": "SBP",
    "Diastolic Blood Pressure": "DBP", "SpO2": "SpO2", "Albumin": "Alb",
    "Bilirubin": "Bili", "Alanine Aminotransferase": "ALT",
    "Aspartate Aminotransferase": "AST", "Alkaline Phosphatase": "ALP",
    "Blood Urea Nitrogen": "BUN", "Creatinine": "Cr", "Chloride": "Cl",
    "Sodium": "Na", "Potassium": "K", "CO2": "CO2", "Platelet Count": "PLT",
}
_SPECIAL = {"AgeInYears": "Age"}

def _abbrev(name):
    if name in _SPECIAL:
        return _SPECIAL[name]
    stat, _, var = name.partition(" ")
    if stat in _STAT and var in _VAR:
        return f"{_STAT[stat]} {_VAR[var]}"
    print("SHAP label not abbreviated:", name)
    return name

def compute_shap(model, X_test, use_gpu=False, device='cuda'):
    if use_gpu:
        import xgboost as xgb
        model.set_params(device=device)      # <-- uses the passed device
        sv = model.get_booster().predict(xgb.DMatrix(X_test), pred_contribs=True)
        return sv[:, :-1]
    import shap
    return np.array(shap.TreeExplainer(model).shap_values(X_test))


def _draw_panel(ax, shap_values, X_test, feature_names, window_size,
                panel_letter, top_k=15, show_xlabel=False, max_points=None):
    n_features=len(feature_names)
    shap_r=np.asarray(shap_values).reshape(X_test.shape[0],window_size,n_features)
    mean_abs=np.mean(np.mean(np.abs(shap_r),axis=1),axis=0)
    top_idx=np.argsort(mean_abs)[-top_k:][::-1]
    top_feats=[feature_names[i] for i in top_idx]
    top_shap=np.mean(shap_r,axis=1)[:,top_idx]
    X_mean=np.mean(X_test.reshape(X_test.shape[0],window_size,-1),axis=1)
    top_vals=X_mean[:,top_idx]
    # Ranking + boxplots (below) use ALL rows. Only the plotted swarm is subsampled,
    # so a dense point cloud stays readable and fast to save. Stats are unchanged.
    n_rows = top_shap.shape[0]
    if max_points and n_rows > max_points:
        sel = np.random.default_rng(0).choice(n_rows, size=max_points, replace=False)
    else:
        sel = slice(None)
    for i,(sv,fv) in enumerate(zip(top_shap.T,top_vals.T)):
        sv_p, fv_p = sv[sel], fv[sel]
        y=i+np.random.uniform(-0.25,0.25,size=len(sv_p))
        lo,hi=np.nanpercentile(fv,[5,95])   # colour scale from FULL data
        ax.scatter(sv_p,y,c=fv_p,cmap=_CMAP,norm=plt.Normalize(lo,hi),
                   alpha=0.8,edgecolors='k',linewidth=0.25,s=4,
                   rasterized=True)   # <-- keeps PDF small; text/boxes stay vector
    for i,sv in enumerate(top_shap.T):
        ax.boxplot(sv,positions=[i],vert=False,widths=0.6,patch_artist=True,
                   boxprops=dict(facecolor='#f0f0f0',alpha=0.65,edgecolor='gray',linewidth=0.4),
                   medianprops=dict(color='black',linewidth=0.4),showfliers=False)
    pretty=[_abbrev(n) for n in top_feats]
    ax.set_yticks(range(top_k)); ax.set_yticklabels(pretty); ax.invert_yaxis()
    ax.tick_params(axis='y', pad=1.5)
    ax.margins(y=0.02)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.6); sp.set_color("black")
    # panel letter: figure-relative-ish small offset so it doesn't drift into neighbour
    ax.text(-0.28,1.03,panel_letter,transform=ax.transAxes,
            fontsize=8,fontweight='bold',va='bottom',ha='right')
    if show_xlabel: ax.set_xlabel('SHAP value (impact on model output)')


def shap_figure(all_shap_inputs, outfile='fig5_shap.pdf',
                order=(8, 24, 48), letters=('a', 'b', 'c'),
                use_gpu=False, top_k=15):
    """Combined a/b-top, c-centered figure saved as a single PDF (+PNG)."""
    fig = plt.figure(figsize=(6.69, 6.69), layout='constrained')
    gs = GridSpec(2, 4, figure=fig, hspace=0.35, wspace=1.1)
    axes = [fig.add_subplot(gs[0, 0:2]),   # a : top-left
            fig.add_subplot(gs[0, 2:4]),   # b : top-right
            fig.add_subplot(gs[1, 1:3])]   # c : bottom-centered

    for ax, hours, letter in zip(axes, order, letters):
        inp = all_shap_inputs[hours]
        fn = list(inp['feature_names'])
        ws = inp.get('window_size', inp['X_test'].shape[1] // len(fn))
        sv = compute_shap(inp['model'], inp['X_test'], use_gpu=use_gpu)
        _draw_panel(ax, sv, inp['X_test'], fn, ws, letter, top_k=top_k,
                    show_xlabel=(letter == letters[-1]))   # xlabel on c only

    fig.savefig(outfile)
    fig.savefig(outfile.replace('.pdf', '.png'), dpi=350)
    plt.show()
    return fig

In [ ]:
shap_cache = {}
for hours in (8, 24, 48):
    inp = all_shap_inputs[hours]
    shap_cache[hours] = compute_shap(inp['model'], inp['X_test'], use_gpu=True, device='cuda:6')
    print(f"{hours}h done: shape {shap_cache[hours].shape}")

In [ ]:
import pickle
with open('shap_cache_rev.pkl', 'wb') as f:
    pickle.dump(shap_cache, f)

In [ ]:
"""
Fig. 5 (SHAP) -- npj-compliant version.

Run this whole file as ONE cell after a kernel restart, with `all_shap_inputs`
already in memory and shap_cache.pkl on disk.

Changes vs the previous version:
  figsize 180x218 mm -> 170x170 mm (was 48 mm over Nature's height cap)
  bbox_inches='tight' removed      (it broke the declared figure width)
  colour map kept as-is -- verified CVD-safe (monotonic lightness, see note below)
  panel letters 9 pt -> 8 pt
  horizontal colourbar added, centred under panel c, so the caption no longer
    has to describe the colours
  line weights 0.6/0.4 -> 1.0/0.8 to match Figs. 4 and S1-S3
  DejaVu fallback removed so a failed Arial lookup is loud

Unchanged by request: the original colour ramp, and a single x-axis label on
panel c only. Because a and b carry no x label, the caption must name the
quantity and note that the three x-ranges differ by an order of magnitude.
"""

# ---------------------------------------------------------------- 1. imports
import pickle
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.font_manager import findfont, FontProperties

# Write files only -- never render inline. A swarm of this many points pushed
# through the notebook websocket as a PNG is what makes the cell appear to hang
# long after the work is finished, and what triggers the server's
# "Starting buffering ..." message.
#
# switch_backend() works at runtime, unlike mpl.use(), so this is safe in a
# kernel where pyplot has already been imported -- no restart needed.
# To get inline plots back later in this session:
#     plt.switch_backend('module://matplotlib_inline.backend_inline')
plt.switch_backend('Agg')
plt.ioff()

# ------------------------------------------- 2. re-register Arial (kernel restart)
FONT_DIR = os.environ.get('SEPSIS_FONT_DIR', '')
for _f in ['arial.ttf', 'arialbd.ttf', 'ariali.ttf', 'arialbi.ttf']:
    if FONT_DIR and os.path.isfile(f'{FONT_DIR}/{_f}'):
        fm.fontManager.addfont(f'{FONT_DIR}/{_f}')
if FONT_DIR:
    for _w, _s in [('normal', 'normal'), ('bold', 'normal')]:
        _p = findfont(FontProperties(family='Arial', weight=_w, style=_s))
        assert 'arial' in _p.lower(), f'Arial {_w}/{_s} did not resolve: {_p}'
print('Arial font configured' if FONT_DIR else 'Using system font fallback')

# ---------------------------------------------------------------- 3. rcParams
plt.rcParams.update({
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],            # no DejaVu fallback
    'font.size': 7, 'axes.labelsize': 7, 'axes.titlesize': 7,
    'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'axes.linewidth': 1.0,
    'xtick.major.width': 1.0, 'ytick.major.width': 1.0,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'figure.dpi': 450, 'savefig.dpi': 450,
})

# ------------------------------------------------------- 4. names and colours
FEATURE_NAME_MAP = {
    "Maximum Pulse": "Max HR", "Maximum Respiratory Rate": "Max RR",
    "Mean Respiratory Rate": "Mean RR",
    "Maximum White Blood Cell Count": "Max WBC", "Maximum Temperature": "Max Temp",
    "Minimum Temperature": "Min Temp", "Minimum White Blood Cell Count": "Min WBC",
    "Maximum Bilirubin": "Max Bili", "Minimum Albumin": "Min Alb",
    "Mean Albumin": "Mean Alb",
    "Mean Alanine Aminotransferase": "Mean ALT",
    "Mean White Blood Cell Count": "Mean WBC",
    "AgeInYears": "Age", "Maximum Albumin": "Max Alb", "Mean Heart Rate": "Mean HR",
    "Minimum Chloride": "Min Cl", "Maximum Alanine Aminotransferase": "Max ALT",
    "Minimum SpO2": "Min SpO2", "Mean Bilirubin": "Mean Bili",
    "Minimum Bilirubin": "Min Bili", "Minimum Alanine Aminotransferase": "Min ALT",
      "Mean Temperature": "Mean Temp", "Maximum Systolic Blood Pressure": "Max SBP", "Mean Pulse": "Mean HR",
}

# Original ramp, kept deliberately. It is a monotonic lightness ramp
# (L* 84 -> 72 -> 62), so it stays orderable under colour-vision deficiency:
# CIELAB dE 61.0 normal, 47.3 deuteranopia (78% retained), 45.3 protanopia (74%).
# The red/green prohibition targets hue-only category contrast, which this is not.
_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "custom_cmap", ["#a6dec1", "#c78a3d"])

_STAT = {"Maximum": "Max", "Minimum": "Min", "Mean": "Mean"}


def _abbrev(name):
    if name in FEATURE_NAME_MAP:
        return FEATURE_NAME_MAP[name]
    parts = name.split()
    if parts and parts[0] in _STAT:
        parts[0] = _STAT[parts[0]]
    out = " ".join(parts)
    return out if len(out) <= 14 else out[:13] + "…"


# --------------------------------------------------------------- 5. one panel
def _draw_panel(ax, shap_values, X_test, feature_names, window_size,
                panel_letter, top_k=15, show_xlabel=True, max_points=None):
    n_features = len(feature_names)
    shap_r = np.asarray(shap_values).reshape(X_test.shape[0], window_size, n_features)
    mean_abs = np.mean(np.mean(np.abs(shap_r), axis=1), axis=0)
    top_idx = np.argsort(mean_abs)[-top_k:][::-1]
    top_feats = [feature_names[i] for i in top_idx]
    top_shap = np.mean(shap_r, axis=1)[:, top_idx]
    X_mean = np.mean(X_test.reshape(X_test.shape[0], window_size, -1), axis=1)
    top_vals = X_mean[:, top_idx]

    n_rows = top_shap.shape[0]
    sel = (np.random.default_rng(0).choice(n_rows, size=max_points, replace=False)
           if (max_points and n_rows > max_points) else slice(None))

    for i, (sv, fv) in enumerate(zip(top_shap.T, top_vals.T)):
        sv_p, fv_p = sv[sel], fv[sel]
        y = i + np.random.uniform(-0.25, 0.25, size=len(sv_p))
        lo, hi = np.nanpercentile(fv, [5, 95])
        ax.scatter(sv_p, y, c=fv_p, cmap=_CMAP, norm=plt.Normalize(lo, hi),
                   alpha=0.8, edgecolors='k', linewidth=0.25, s=4,
                   rasterized=True)

    for i, sv in enumerate(top_shap.T):
        ax.boxplot(sv, positions=[i], vert=False, widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor='#f0f0f0', alpha=0.65,
                                 edgecolor='gray', linewidth=0.8),
                   medianprops=dict(color='black', linewidth=0.8),
                   showfliers=False)

    ax.set_yticks(range(top_k))
    ax.set_yticklabels([_abbrev(n) for n in top_feats])
    ax.invert_yaxis()
    ax.tick_params(axis='y', pad=1.5)
    ax.margins(y=0.02)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(1.0); sp.set_color('black')

    ax.text(-0.26, 1.03, panel_letter, transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='bottom', ha='right')

    if show_xlabel:
        ax.set_xlabel('SHAP value (impact on model output)')


# ------------------------------------------------------------ 6. whole figure
def shap_figure_from_cache(shap_cache, all_shap_inputs, outfile='fig5_shap.pdf',
                           order=(8, 24, 48), letters=('a', 'b', 'c'),
                           top_k=15, save_pdf=True, save_png=True, dpi=450):
    """Draw Fig. 5 from pre-computed SHAP values. 170 x 170 mm, scale 1.0."""
    fig = plt.figure(figsize=(6.69, 6.69), layout='constrained')
    # third row is a thin strip for the horizontal colourbar, spanning the same
    # two middle columns as panel c so it sits centred underneath it
    gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 0.06])
    axes = [fig.add_subplot(gs[0, 0:2]),
            fig.add_subplot(gs[0, 2:4]),
            fig.add_subplot(gs[1, 1:3])]
    cax = fig.add_subplot(gs[2, 1:3])

    for ax, hours, letter in zip(axes, order, letters):
        inp = all_shap_inputs[hours]
        fn = list(inp['feature_names'])
        ws = inp.get('window_size', inp['X_test'].shape[1] // len(fn))
        _draw_panel(ax, shap_cache[hours], inp['X_test'], fn, ws, letter,
                    top_k=top_k,
                    show_xlabel=(letter == letters[-1]),   # x label on c only
                    max_points=None)

    # colour key in the figure, so the caption does not have to describe colours
    sm = mpl.cm.ScalarMappable(cmap=_CMAP, norm=plt.Normalize(0, 1))
    cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
    cb.set_label('Feature value')
    cb.set_ticks([0, 1]); cb.set_ticklabels(['Low', 'High'])
    cb.outline.set_linewidth(0.6)
    cax.tick_params(length=0, pad=1.5)

    # if the panels look cramped, loosen here rather than with tight_layout
    try:
        fig.get_layout_engine().set(w_pad=0.02, h_pad=0.02, wspace=0.10, hspace=0.06)
    except AttributeError:
        pass

    if save_png:
        fig.savefig(outfile.replace('.pdf', '.png'), dpi=dpi)   # no bbox_inches
    if save_pdf:
        fig.savefig(outfile, dpi=dpi)                           # no bbox_inches
    plt.close(fig)          # release it; do NOT return the figure to the notebook
    return None


# --------------------------------------------------------------------- 7. run
with open('shap_cache_rev.pkl', 'rb') as f:
    shap_cache = pickle.load(f)

# Use all_shap_inputs if it is in memory; otherwise fall back to the slim pickle,
# so this file also runs standalone in a fresh kernel with no pipeline re-run.
try:
    all_shap_inputs
except NameError:
    with open('shap_inputs_slim_rev.pkl', 'rb') as f:
        all_shap_inputs = pickle.load(f)
    print('loaded inputs from shap_inputs_slim_rev.pkl')

for h in (8, 24, 48):                      # rows of the cache must match X_test
    n_cache = shap_cache[h].shape[0]
    n_x     = all_shap_inputs[h]['X_test'].shape[0]
    assert n_cache == n_x, f'{h}h: cache has {n_cache} rows, X_test has {n_x}'

shap_figure_from_cache(shap_cache, all_shap_inputs, outfile='fig5_shap.pdf')

# --------------------------------------------- 8. save the inputs for next time
import os
if not os.path.exists('shap_inputs_slim_rev.pkl'):
    slim = {}
    for h in (8, 24, 48):
        inp = all_shap_inputs[h]
        fn = list(inp['feature_names'])
        slim[h] = {'X_test': inp['X_test'], 'feature_names': fn,
                   'window_size': inp.get('window_size',
                                          inp['X_test'].shape[1] // len(fn))}
    with open('shap_inputs_slim_rev.pkl', 'wb') as f:
        pickle.dump(slim, f)
    print('wrote shap_inputs_slim_rev.pkl')

print('wrote fig5_shap.pdf, fig5_shap.png')

In [ ]:
# =============================================================================
# REVIEWER EXPERIMENTS (colleague's package, run inside this notebook - no export file)
# Full STRIDE vs STRIDE without SIRS variables, same splits / imputation / scaling /
# hyperparameters. Run in the same kernel after run_all_windows().
# =============================================================================
import re
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier

OUTPUT = Path('stride_reviewer_results'); OUTPUT.mkdir(exist_ok=True)
TARGET_SENS = 0.80
MODEL_PARAMS = dict(n_estimators=100, max_depth=12, random_state=0, n_jobs=64)   # identical to the main model
SIRS_PATTERNS = [r'(^|[_\W])(pulse|heart.?rate|hr)([_\W]|$)',          # colleague's patterns
                 r'(^|[_\W])(resp|respiratory.?rate|rr)([_\W]|$)',
                 r'(^|[_\W])(temp|temperature)([_\W]|$)',
                 r'(^|[_\W])(wbc|white.?blood)([_\W]|$)']
GROUPS = {'septic': septic_encounters,
          'septic_vs_non_septic_sirs_pos': sirs_pos_encounters,
          'septic_vs_non_septic_sirs_neg': sirs_neg_encounters}

def threshold_for_sensitivity(y, p, target=TARGET_SENS):
    """Highest cutoff that still reaches the target sensitivity (colleague's function)."""
    pos = np.sort(p[y == 1])
    return float(pos[max(0, min(len(pos) - 1, int(np.floor((1 - target) * len(pos)))))])

def metrics_from_pred(y, p, pred):
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    div = lambda a, b: a / b if b else np.nan
    two = len(np.unique(y)) == 2
    return dict(n=len(y), positive=int(y.sum()), negative=int(len(y) - y.sum()),
                TP=tp, FP=fp, TN=tn, FN=fn,
                sensitivity=div(tp, tp + fn), specificity=div(tn, tn + fp),
                PPV=div(tp, tp + fp), NPV=div(tn, tn + fn), F1=div(2 * tp, 2 * tp + fp + fn),
                accuracy=div(tp + tn, len(y)),
                ROC_AUC=roc_auc_score(y, p) if two else np.nan,
                average_precision=average_precision_score(y, p) if two else np.nan,
                Brier=brier_score_loss(y, p))

def alert_stats(enc, y, pred):
    codes, _ = pd.factorize(enc)
    flags = np.bincount(codes, weights=pred.astype(float))
    septic = np.bincount(codes, weights=y.astype(float)) > 0
    return dict(windows=len(pred), flags=int(pred.sum()), encounters=len(flags),
                encounters_flagged=int((flags > 0).sum()),
                septic_encounters_flagged_pct=100 * (flags[septic] > 0).mean() if septic.any() else np.nan,
                non_septic_encounters_flagged_pct=100 * (flags[~septic] > 0).mean() if (~septic).any() else np.nan,
                repeated_flags=int(np.maximum(flags - 1, 0).sum()),
                flags_per_1000_windows=1000 * pred.mean(),
                flags_per_1000_patient_hours=1000 * pred.mean() / 8)   # each prediction = one 8-hour interval

rows, subrows, alertrows = [], [], []
ABL_PRED = {}                                             # kept for bootstrap CIs after the run
E2P = {df['EncounterKey'].iloc[0]: df['PatientDurableKey'].iloc[0] for df in filtered_data}
for hours, w in WINDOW_SIZES.items():
    splits = build_bootstrapped_splits(w)                 # cached splits: no new imputation
    names = splits[0]['feature_names']
    base = np.array([any(re.search(pt, str(f), re.I) for pt in SIRS_PATTERNS) for f in names])
    sirs_cols = np.tile(base, w)                          # every 8-hour segment of the window
    print(f"\n{hours}h: removing {base.sum()} variables x {w} segments: {[n for n, b in zip(names, base) if b]}")
    assert base.sum() >= 4, "Fewer SIRS variables than expected - check feature names"

    for i, s in enumerate(splits):
        y = {c: np.asarray(s['y_' + c]).astype(int) for c in ('train', 'test', 'val')}
        enc = {c: np.asarray(s[c + '_encounter_keys']) for c in ('test', 'val')}
        # test patients split in two halves (by encounter) for the cross-fitted threshold
        upat, inv = np.unique([E2P[e] for e in enc['test']], return_inverse=True)   # halves BY PATIENT
        half_a = (np.random.RandomState(100 + i).rand(len(upat)) < 0.5)[inv]       # same halves as PAPER NUMBERS

        for model_name, keep in (('Full STRIDE', np.ones(len(sirs_cols), bool)), ('Without SIRS', ~sirs_cols)):
            model = XGBClassifier(**MODEL_PARAMS).fit(s['X_train'][:, keep], y['train'])
            p = {c: model.predict_proba(s['X_' + c][:, keep])[:, 1] for c in ('train', 'test', 'val')}
            if model_name == 'Full STRIDE':                  # consistency check with the main run
                print(f"  split {i}: full-model test AUC {roc_auc_score(y['test'], p['test']):.4f} "
                      f"(main run: {all_results[hours]['overall_aucs'][i]:.4f})")

            t_train = threshold_for_sensitivity(y['train'], p['train'])
            t_test = threshold_for_sensitivity(y['test'], p['test'])
            t_a = threshold_for_sensitivity(y['test'][half_a], p['test'][half_a])
            t_b = threshold_for_sensitivity(y['test'][~half_a], p['test'][~half_a])
            preds = {
                # colleague's method: threshold from the model's own training predictions
                ('test', 'training-set threshold'): p['test'] >= t_train,
                ('val', 'training-set threshold'): p['val'] >= t_train,
                # held-out method: test = cross-fitted halves; validation = threshold from internal test
                ('test', 'held-out threshold'): np.where(half_a, p['test'] >= t_b, p['test'] >= t_a),
                ('val', 'held-out threshold'): p['val'] >= t_test,
            }
            thr_used = {'training-set threshold': t_train, 'held-out threshold': t_test}
            for (cohort, method), pred in preds.items():
                key = dict(hours=hours, partition=i, model=model_name, cohort=cohort, threshold_method=method)
                rows.append({**key, **metrics_from_pred(y[cohort], p[cohort], pred), 'threshold': thr_used[method]})
                alertrows.append({**key, **alert_stats(enc[cohort], y[cohort], pred)})
                if cohort == 'test':
                    for g, keys in GROUPS.items():
                        m = np.isin(enc['test'], list(keys))
                        subrows.append({**key, 'group': g, **metrics_from_pred(y['test'][m], p['test'][m], pred[m])})
            ABL_PRED[(hours, i, model_name)] = {c: p[c].astype('float32') for c in ('test', 'val')}
            del model, p
        print(f"  {hours}h split {i + 1}/{len(splits)} done")
    del splits

metrics = pd.DataFrame(rows); subgroups = pd.DataFrame(subrows); alerts = pd.DataFrame(alertrows)
cols = ['ROC_AUC', 'average_precision', 'sensitivity', 'specificity', 'PPV', 'NPV', 'F1', 'Brier']
sub_cols = ['n', 'positive', 'negative', 'ROC_AUC', 'sensitivity', 'specificity', 'PPV']
for method, tag in (('training-set threshold', 'colleague_training_threshold'),
                    ('held-out threshold', 'heldout_threshold')):
    m_, s_, a_ = (df[df['threshold_method'] == method] for df in (metrics, subgroups, alerts))
    m_.to_csv(OUTPUT / f'ablation_metrics_by_partition_{tag}.csv', index=False)
    s_.to_csv(OUTPUT / f'subgroups_counts_and_metrics_{tag}.csv', index=False)
    a_.to_csv(OUTPUT / f'alert_frequency_{tag}.csv', index=False)
    m_.groupby(['hours', 'cohort', 'model'], sort=False)[cols].agg(['mean', 'std']).to_csv(
        OUTPUT / f'ablation_summary_{tag}.csv')
    print(f"\n{'=' * 90}\n{method.upper()}  (80% sensitivity target)\n{'=' * 90}")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
        display(m_.groupby(['hours', 'cohort', 'model'], sort=False)[cols].mean().round(3))
        display(s_.groupby(['hours', 'group', 'model'], sort=False)[sub_cols].mean().round(3))

In [ ]:
import os
print(sorted(os.listdir('stride_reviewer_results')))
print('ABL_PRED entries:', len(ABL_PRED))          # should be 30 (3 windows x 5 splits x 2 models)

cols = ['ROC_AUC', 'average_precision', 'sensitivity', 'specificity', 'PPV', 'NPV', 'F1', 'Brier']
sub_cols = ['n', 'positive', 'negative', 'ROC_AUC', 'sensitivity', 'specificity', 'PPV']
for method in ('held-out threshold', 'training-set threshold'):
    print(f"\n===== {method.upper()} =====")
    m_ = metrics[metrics['threshold_method'] == method]
    s_ = subgroups[subgroups['threshold_method'] == method]
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
        display(m_.groupby(['hours', 'cohort', 'model'], sort=False)[cols].mean().round(3))
        display(s_.groupby(['hours', 'group', 'model'], sort=False)[sub_cols].mean().round(3))

In [ ]:
import xgboost, sklearn
print("xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
der = {df['EncounterKey'].iloc[0]: df['PatientDurableKey'].iloc[0] for df in filtered_data}
groups = {'Septic': septic_encounters,
          'Non-septic SIRS+': set(non_septic_sirs_pos['EncounterKey']),
          'Non-septic SIRS-': set(non_septic_sirs_neg['EncounterKey'])}
for g, encs in groups.items():
    e = [k for k in encs if k in der]
    print(f"{g:17} encounters {len(e):,} | patients {len(set(der[k] for k in e)):,}")
print(f"{'All derivation':17} encounters {len(der):,} | patients {len(set(der.values())):,}")

v = {}
for df in filtered_patients_data_val:
    k = df['EncounterKey'].iloc[0]
    v.setdefault(k, [df['PatientDurableKey'].iloc[0], 0])
    v[k][1] = max(v[k][1], int(df['Label'].max()))
for lab, name in ((1, 'Adjudicated septic'), (0, 'Adjudicated non-septic')):
    ks = [k for k, (p, l) in v.items() if l == lab]
    print(f"{name:22} encounters {len(ks)} | patients {len(set(v[k][0] for k in ks))}")

In [ ]:
llm = rs2.copy()
llm['EncounterKey'] = pd.to_numeric(llm['EncounterKey'], errors='coerce')
llm = llm.dropna(subset=['EncounterKey'])
print("Llama outputs:", len(llm), "| encounters with >1 output:", llm['EncounterKey'].duplicated().sum())
llm = llm.drop_duplicates('EncounterKey')

# derivation: how many initially septic encounters were assessed, and how many were reclassified
der_sep = {df['EncounterKey'].iloc[0] for df in filtered_patients_data} | set(non_septic_encs)
d = llm[llm['EncounterKey'].isin(der_sep)]['Llama_Label'].value_counts()
print(f"Derivation septic encounters: {len(der_sep):,} | with a Llama output: {d.sum():,} | 'No' (reclassified): {d.get('No', 0):,}")

# adjudicated cohort: agreement with the physicians
print("Physician labels in the file:", ValCohort['Label'].unique())
m = ValCohort.merge(llm, on='EncounterKey', how='left')
print("Adjudicated encounters:", len(m), "| with a Llama output:", m['Llama_Label'].isin(['Yes', 'No']).sum())
m = m[m['Llama_Label'].isin(['Yes', 'No'])]
if len(m):
    y = (m['Label'] == 'Yes').to_numpy()        # physician: sepsis confirmed
    p = (m['Llama_Label'] == 'Yes').to_numpy()  # Llama: sepsis supported
    tp, fp, fn, tn = (y & p).sum(), (~y & p).sum(), (y & ~p).sum(), (~y & ~p).sum()
    def wilson(k, n, z=1.96):
        if n == 0: return (np.nan, np.nan, np.nan)
        f = k / n; c = z * np.sqrt(f * (1 - f) / n + z**2 / (4 * n**2)); d = 1 + z**2 / n
        return f, (f + z**2 / (2 * n) - c) / d, (f + z**2 / (2 * n) + c) / d
    print(f"TP {tp}  FP {fp}  FN {fn}  TN {tn}")
    for name, k, n in [('Accuracy', tp + tn, len(y)), ('Sensitivity', tp, tp + fn), ('Specificity', tn, tn + fp),
                       ('PPV', tp, tp + fp), ('NPV', tn, tn + fn)]:
        f, lo, hi = wilson(k, n)
        print(f"{name:12} {k}/{n} = {f:.3f} (95% CI {lo:.3f}-{hi:.3f})")
    print(f"F1 {2 * tp / (2 * tp + fp + fn):.3f}")

In [ ]:
# =============================================================================
# PAPER NUMBERS - run in the same kernel, after run_all_windows(). No retraining.
#  A) Discrimination with patient-level bootstrap 95% CIs (ROC-AUC, AP, Brier; subgroups)
#  B) Operating points at 80/85/90% sensitivity for STRIDE, SOFA, Epic (held-out thresholds,
#     test halves split BY PATIENT) + SIRS at its rule, with patient-level bootstrap CIs
#  C) Missing-data rates after interpolation (Supplementary table)
#  D) Epic-eligible population counts
# Outputs: paper_discrimination.csv, paper_operating_points.csv, paper_missingness.csv, paper_epic_population.csv
# =============================================================================
from sklearn.metrics import roc_auc_score, average_precision_score

B_PER_SPLIT = 200                       # bootstrap resamples per split (x5 splits = 1,000 per CI)
TARGETS = (0.80, 0.85, 0.90)
keys = ['EncounterKey', 'DateTimeKey']
_RULE = {'sirs': (0, 1, 'Sirs'), 'sofa': (2, 3, 'NormalizedSofaScore'), 'epic': (4, 5, 'NormalizedEpicScore')}
NAMES = {'model': 'STRIDE', 'sirs': 'SIRS', 'sofa': 'SOFA', 'epic': 'Epic'}
GROUPS = {'Septic subset': septic_encounters, 'SIRS+ comparison': sirs_pos_encounters,
          'SIRS- comparison': sirs_neg_encounters}

enc_to_pat = {df['EncounterKey'].iloc[0]: df['PatientDurableKey'].iloc[0]
              for df in list(filtered_data) + list(filtered_patients_data_val)}

def _window_keys(d, w):
    ln = d['lengths']; starts = np.concatenate([[0], np.cumsum(ln)[:-1]])
    nw = np.clip(ln - w + 1, 0, None)
    ws = np.repeat(starts, nw) + (np.arange(nw.sum()) - np.repeat(np.cumsum(nw) - nw, nw))
    return pd.DataFrame({'EncounterKey': d['enc'][ws].tolist(), 'DateTimeKey': d['dtk'][ws + w - 1].tolist(),
                         'Label': d['y'][ws + w - 1]})

def _sofa_points(s):
    """Undo the per-cohort min-max scaling of SOFA so test thresholds transfer to the adjudicated cohort."""
    pos = s[s > 0]
    if s.min() != 0 or len(pos) == 0:
        return s, False
    mx = round(1 / pos.min())
    pts = s * mx
    return (np.round(pts), True) if np.allclose(pts, np.round(pts), atol=1e-6) else (s, False)

def scored(h, w, i, cohort, method):
    """enc, patient, y, score for one split, aligned with all_curve_data (checked)."""
    wk = _window_keys(_split_cache[i][cohort], w)
    cd = all_curve_data[h][cohort][method]
    if method == 'model':
        wk['score'] = np.asarray(cd['y_score'][i], dtype=float)
    else:
        t, v, col = _RULE[method]
        wk = wk.merge(_rule_cache[i][t if cohort == 'test' else v], on=keys, how='left').dropna()
        wk = wk.rename(columns={col: 'score'})
    assert np.array_equal(wk['Label'].to_numpy(), np.asarray(cd['y_true'][i])), (h, i, cohort, method)
    s = wk['score'].to_numpy(dtype=float)
    if method == 'sofa':
        s, ok = _sofa_points(s)
        if not ok:
            print(f"  note: SOFA points not recoverable ({h}h split {i} {cohort}); thresholds stay on the scaled score")
    enc = wk['EncounterKey'].to_numpy()
    pat = np.array([enc_to_pat[e] for e in enc])
    return enc, pat, wk['Label'].to_numpy().astype(int), s

class Ranked:
    """Score ranks computed once, so weighted AUC / AP per bootstrap resample are O(n)."""
    def __init__(self, y, s, cal=None):
        self.y = y.astype(float); self.s = s
        self.u, self.inv = np.unique(s, return_inverse=True); self.K = len(self.u)
        self.cal = s if cal is None else cal                                  # 0-1 score for Brier / ECE
        self.bin = np.searchsorted(np.linspace(0, 1, 11)[1:-1], self.cal)     # same 10 bins as the figures
    def auc(self, w):
        P = np.bincount(self.inv, w * self.y, self.K); N = np.bincount(self.inv, w * (1 - self.y), self.K)
        return float((P * (np.cumsum(N) - N + 0.5 * N)).sum() / (P.sum() * N.sum()))
    def ap(self, w):
        P = np.bincount(self.inv, w * self.y, self.K)[::-1]; N = np.bincount(self.inv, w * (1 - self.y), self.K)[::-1]
        tp, fp = np.cumsum(P), np.cumsum(N)
        prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
        return float((P / P.sum() * prec).sum())
    def brier(self, w):
        return float((w * (self.cal - self.y) ** 2).sum() / w.sum())
    def ece(self, w):
        W = np.bincount(self.bin, w, 10); Y = np.bincount(self.bin, w * self.y, 10)
        S = np.bincount(self.bin, w * self.cal, 10)
        return float(np.abs(Y - S).sum() / W.sum())

def op_metrics(y, pred, w):
    yb = y == 1
    tp = (w * (pred & yb)).sum(); fp = (w * (pred & ~yb)).sum()
    fn = (w * (~pred & yb)).sum(); tn = (w * (~pred & ~yb)).sum()
    div = lambda a, b: a / b if b else np.nan
    return {'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn, 'Sensitivity': div(tp, tp + fn), 'Specificity': div(tn, tn + fp),
            'PPV': div(tp, tp + fp), 'NPV': div(tn, tn + fn), 'F1': div(2 * tp, 2 * tp + fp + fn),
            'Accuracy': div(tp + tn, tp + tn + fp + fn)}

def thr_for(y, s, target):
    pos = np.sort(s[y == 1])[::-1]
    return pos[int(np.ceil(target * len(pos))) - 1]

CAL = ('model', 'sofa', 'epic') 
OP_STATS = ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1', 'Accuracy']
disc, ops = defaultdict(lambda: {'point': [], 'boot': []}), defaultdict(lambda: {'point': [], 'boot': [], 'thr': [], 'n': []})

for h, w in WINDOW_SIZES.items():
    for i in tqdm(sorted(_split_cache), desc=f'{h}h splits'):
        rng = np.random.default_rng(1000 * h + i)
        data = {(c, m): scored(h, w, i, c, m) for c in ('test', 'val') for m in ('model', 'sirs', 'sofa', 'epic')}
        # one bootstrap weight vector per patient and cohort, shared by all methods (paired resamples)
        pats = {c: np.unique(data[(c, 'model')][1]) for c in ('test', 'val')}
        idx = {k: np.searchsorted(pats[k[0]], v[1]) for k, v in data.items()}
        # patient halves of the test set for the cross-fitted thresholds
        half = (np.random.RandomState(100 + i).rand(len(pats['test'])) < 0.5)
        plan = {}
        for (c, m), (enc, pat, y, s) in data.items():
            cal = np.asarray(all_curve_data[h][c][m]['y_score'][i], dtype=float)   # original 0-1 score
            rk = Ranked(y, s, cal)
            sub = {}
            if c == 'test':
                for g, ks in GROUPS.items():
                    msk = np.isin(enc, list(ks))
                    sub[g] = (msk, Ranked(y[msk], s[msk]))
            preds = {}
            if m == 'sirs':
                preds['SIRS >= 2'] = s >= 1
            else:
                for t in TARGETS:
                    if c == 'test':
                        a = half[idx[(c, m)]]
                        ta, tb = thr_for(y[a], s[a], t), thr_for(y[~a], s[~a], t)
                        preds[f'Sens >= {t:.0%}'] = (np.where(a, s >= tb, s >= ta), (ta + tb) / 2)
                    else:
                        yt, st = data[('test', m)][2], data[('test', m)][3]
                        tt = thr_for(yt, st, t)
                        preds[f'Sens >= {t:.0%}'] = (s >= tt, tt)
            plan[(c, m)] = (y, s, rk, sub, preds)
            ones = np.ones(len(y))
            key = (h, c, NAMES[m], 'Overall')
            disc[key]['point'].append([rk.auc(ones), rk.ap(ones),
                                       rk.brier(ones) if m in CAL else np.nan, rk.ece(ones) if m in CAL else np.nan])
            for g, (msk, srk) in sub.items():
                disc[(h, c, NAMES[m], g)]['point'].append([srk.auc(np.ones(msk.sum())), np.nan, np.nan, np.nan])
            for op, pr in preds.items():
                pred, thr = pr if isinstance(pr, tuple) else (pr, 1.0)
                o = ops[(h, c, NAMES[m], op)]
                o['point'].append(op_metrics(y, pred, ones)); o['thr'].append(thr)
                o['n'].append((int(y.sum()), int(len(y) - y.sum()), len(np.unique(enc))))
        for b in range(B_PER_SPLIT):
            wp = {c: rng.multinomial(len(p), np.full(len(p), 1 / len(p))).astype(float) for c, p in pats.items()}
            for (c, m), (y, s, rk, sub, preds) in plan.items():
                wr = wp[c][idx[(c, m)]]
                disc[(h, c, NAMES[m], 'Overall')]['boot'].append([rk.auc(wr), rk.ap(wr),
                                        rk.brier(wr) if m in CAL else np.nan, rk.ece(wr) if m in CAL else np.nan])
                for g, (msk, srk) in sub.items():
                    disc[(h, c, NAMES[m], g)]['boot'].append([srk.auc(wr[msk]), np.nan, np.nan, np.nan])
                for op, pr in preds.items():
                    pred = pr[0] if isinstance(pr, tuple) else pr
                    ops[(h, c, NAMES[m], op)]['boot'].append(op_metrics(y, pred, wr))

# sanity check: the weighted AUC with unit weights equals sklearn's on one series
_y, _s = plan[('test', 'model')][0], plan[('test', 'model')][1]
assert np.isclose(Ranked(_y, _s).auc(np.ones(len(_y))), roc_auc_score(_y, _s))
assert np.isclose(Ranked(_y, _s).ap(np.ones(len(_y))), average_precision_score(_y, _s))
assert np.isclose(Ranked(_y, _s).ece(np.ones(len(_y))), functions_changes.calibration_error(_y, _s, 10, 'uniform')[0])

fmt = lambda m, lo, hi, d=3: f"{m:.{d}f} ({lo:.{d}f}-{hi:.{d}f})"
rows = []
for (h, c, m, g), v in disc.items():
    pt, bt = np.nanmean(np.array(v['point'], float), axis=0), np.array(v['boot'], float)
    r = {'Window': f'{h}h', 'Cohort': 'Internal test' if c == 'test' else 'Adjudicated', 'Method': m, 'Group': g}
    for j, k in enumerate(['ROC-AUC', 'Average precision', 'Brier', 'ECE']):
        if not np.isnan(pt[j]):
            lo, hi = np.nanpercentile(bt[:, j], [2.5, 97.5]); r[k] = fmt(pt[j], lo, hi)
    rows.append(r)
discrimination = pd.DataFrame(rows)
discrimination.to_csv('paper_discrimination.csv', index=False)

rows = []
for (h, c, m, op), v in ops.items():
    pt = pd.DataFrame(v['point']).mean(); bt = pd.DataFrame(v['boot']); n = np.mean(v['n'], axis=0)
    r = {'Window': f'{h}h', 'Cohort': 'Internal test' if c == 'test' else 'Adjudicated', 'Method': m,
         'Operating point': op, 'Threshold': round(float(np.mean(v['thr'])), 4),
         'N positive': int(round(n[0])), 'N negative': int(round(n[1])), 'Encounters': int(round(n[2]))}
    for k in ('TP', 'FP', 'TN', 'FN'):
        r[k] = int(round(pt[k]))
    for k in OP_STATS:
        lo, hi = np.nanpercentile(bt[k], [2.5, 97.5]); r[k] = fmt(pt[k], lo, hi)
    rows.append(r)
operating_points = pd.DataFrame(rows)
operating_points.to_csv('paper_operating_points.csv', index=False)

# ---- C) missingness after interpolation (interval level, before multivariable imputation)
VARS = {'Heart rate': 'Mean Pulse', 'Respiratory rate': 'Mean Respiratory Rate',
        'Systolic BP': 'Mean Systolic Blood Pressure', 'Diastolic BP': 'Mean Diastolic Blood Pressure',
        'Temperature': 'Mean Temperature', 'Oxygen saturation': 'Mean SpO2', 'Bilirubin': 'Mean Bilirubin',
        'Alanine aminotransferase': 'Mean Alanine Aminotransferase',
        'Aspartate aminotransferase': 'Mean Aspartate Aminotransferase',
        'Alkaline phosphatase': 'Mean Alkaline Phosphatase', 'Blood urea nitrogen': 'Mean Blood Urea Nitrogen',
        'Creatinine': 'Mean Creatinine', 'Platelets': 'Mean Platelet Count', 'Potassium': 'Mean Potassium',
        'Sodium': 'Mean Sodium', 'Chloride': 'Mean Chloride', 'Albumin': 'Mean Albumin',
        'White blood cell count': 'Mean White Blood Cell Count', 'CO2': 'Mean CO2'}
der_encs = {df['EncounterKey'].iloc[0] for df in filtered_data}
pos_set, neg_set = set(non_septic_sirs_pos['EncounterKey']), set(non_septic_sirs_neg['EncounterKey'])
val_lab = {}
for df in filtered_patients_data_val:
    k = df['EncounterKey'].iloc[0]; val_lab[k] = max(val_lab.get(k, 0), int(df['Label'].max()))
der_pos, der_neg = der_encs & pos_set, der_encs & neg_set
pools = {'Septic (derivation)': [df for df in filtered_patients_data if df['EncounterKey'].iloc[0] in der_encs],
         'Non-septic SIRS+ (derivation)': [df for df in filtered_patients_data_non if df['EncounterKey'].iloc[0] in der_pos],
         'Non-septic SIRS- (derivation)': [df for df in filtered_patients_data_non if df['EncounterKey'].iloc[0] in der_neg],
         'Septic (adjudicated)': [df for df in filtered_patients_data_val if val_lab[df['EncounterKey'].iloc[0]] == 1],
         'Non-septic (adjudicated)': [df for df in filtered_patients_data_val if val_lab[df['EncounterKey'].iloc[0]] == 0]}
miss = {}
for name, dfs in pools.items():
    big = pd.concat([df[[c for c in VARS.values() if c in df.columns]] for df in dfs], ignore_index=True)
    miss[name] = {v: f"{100 * big[c].isna().mean():.1f}%" if c in big else 'n/a' for v, c in VARS.items()}
missingness = pd.DataFrame(miss)
missingness.to_csv('paper_missingness.csv')

# ---- D) Epic-eligible population (>= 1 recorded Epic score in an analysed interval)
try:
    sc = functions_changes.process_epic_scores(epic_total, pd.concat(list(filtered_data) + list(filtered_patients_data_val), ignore_index=True))
    epic_encs = set(sc.loc[sc['NormalizedEpicScore'].notna(), 'EncounterKey'])
    rows = []
    for name, dfs in pools.items():
        ks = {df['EncounterKey'].iloc[0] for df in dfs}
        e = ks & epic_encs
        rows.append({'Group': name, 'Encounters': len(ks), 'With Epic score': len(e),
                     'Patients with Epic score': len({enc_to_pat[k] for k in e})})
    epic_population = pd.DataFrame(rows)
    epic_population.to_csv('paper_epic_population.csv', index=False)
except Exception as ex:
    epic_population = None
    print("Epic population part skipped:", ex)

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
    display(discrimination)
    display(operating_points[operating_points['Operating point'].isin(['Sens >= 80%', 'SIRS >= 2'])])
    display(missingness)
    if epic_population is not None:
        display(epic_population)

In [ ]:
# =============================================================================
# TABLE 1 + SUPPLEMENTARY "EPIC POPULATION" TABLE  (run AFTER paper_numbers_cell.py)
# Encounter level: demographics from PatientDim, admission type and length of stay from
# EncounterFact, vitals and labs = per-encounter mean of the 8-hour means (recorded +
# interpolated values, before multivariable imputation), then mean (SD) across encounters.
# Output: table1.csv, table1_epic_population.csv
# =============================================================================
def _sql_chunks(template, keys, size=30000):
    keys = [int(k) for k in keys]
    return pd.concat([do_hhc_sql(template.format(", ".join(map(str, keys[j:j + size]))))
                      for j in range(0, len(keys), size)], ignore_index=True)

all_encs = {k for dfs in pools.values() for k in (df['EncounterKey'].iloc[0] for df in dfs)}
enc_pat = {k: enc_to_pat[k] for k in all_encs}

_pat_keys = {int(v) for v in enc_pat.values()}
try:      # BirthDate gives age at admission; fall back to AgeInYears if the column is not there
    pat = _sql_chunks("SELECT DurableKey, Sex, BirthDate, AgeInYears, Ethnicity, FirstRace, SmokingStatus "
                      "FROM CAB_EPIC.dbo.PatientDim WHERE DurableKey IN ({})", _pat_keys)
except Exception as ex:
    print("BirthDate not available, using AgeInYears:", str(ex)[:120])
    pat = _sql_chunks("SELECT DurableKey, Sex, AgeInYears, Ethnicity, FirstRace, SmokingStatus "
                      "FROM CAB_EPIC.dbo.PatientDim WHERE DurableKey IN ({})", _pat_keys)
pat['DurableKey'] = pat['DurableKey'].astype('int64')
pat = pat.drop_duplicates('DurableKey').set_index('DurableKey')
enc = _sql_chunks("SELECT EncounterKey, AdmissionDateKey, DischargeDateKey, AdmissionType "
                  "FROM CAB_EPIC.dbo.EncounterFact WHERE EncounterKey IN ({})", all_encs)
enc['EncounterKey'] = enc['EncounterKey'].astype('int64')
enc = enc.drop_duplicates('EncounterKey').set_index('EncounterKey')
adm = pd.to_datetime(enc['AdmissionDateKey'].astype(str), format='%Y%m%d', errors='coerce')
dis = pd.to_datetime(enc['DischargeDateKey'].astype(str), format='%Y%m%d', errors='coerce')
enc['LOS'] = (dis - adm).dt.days
enc['AdmissionDate'] = adm

def race(x):
    x = str(x)
    if x.startswith('White'): return 'White or Caucasian'
    if x.startswith('Black'): return 'Black or African American'
    if x in ('nan', 'None', '', '*Unspecified', 'Unknown', 'Patient Refused', 'Declined', '*Unknown'): return 'Unknown'
    return 'Other'
def ethnicity(x):
    x = str(x).lower()
    if 'not hispanic' in x or 'non-hispanic' in x: return 'Not Hispanic or Latino'
    if 'hispanic' in x or 'latino' in x: return 'Hispanic or Latino'
    return 'Unknown'
def smoking(x):
    x = str(x).lower()
    if 'never assessed' in x: return 'Never assessed'
    if 'never' in x: return 'Never smoker'
    if 'former' in x: return 'Former smoker'
    if any(t in x for t in ('current', 'every day', 'some day', 'light', 'heavy', 'smoker')): return 'Current smoker'
    return 'Other / unknown'
def admission(x):
    x = str(x)
    return x if x in ('Emergency', 'Urgent', 'Elective') else 'Other'

VITALS = {'Heart rate (bpm)': 'Mean Pulse', 'Respiratory rate (breaths/min)': 'Mean Respiratory Rate',
          'Systolic BP (mmHg)': 'Mean Systolic Blood Pressure', 'Diastolic BP (mmHg)': 'Mean Diastolic Blood Pressure',
          'Temperature (F)': 'Mean Temperature', 'Oxygen saturation (%)': 'Mean SpO2'}
LABS = {'Bilirubin (mg/dL)': 'Mean Bilirubin', 'Alanine aminotransferase (U/L)': 'Mean Alanine Aminotransferase',
        'Aspartate aminotransferase (U/L)': 'Mean Aspartate Aminotransferase',
        'Alkaline phosphatase (U/L)': 'Mean Alkaline Phosphatase', 'Blood urea nitrogen (mg/dL)': 'Mean Blood Urea Nitrogen',
        'Creatinine (mg/dL)': 'Mean Creatinine', 'Platelets (x10^9/L)': 'Mean Platelet Count',
        'Potassium (mmol/L)': 'Mean Potassium', 'Sodium (mmol/L)': 'Mean Sodium', 'Chloride (mmol/L)': 'Mean Chloride',
        'Albumin (g/dL)': 'Mean Albumin', 'White blood cell count (x10^9/L)': 'Mean White Blood Cell Count',
        'CO2 (mmol/L)': 'Mean CO2'}
OTHER = {'Height (cm)': 'Mean Height', 'Weight (kg)': 'Mean Weight'}
num_cols = list(VITALS.values()) + list(LABS.values()) + list(OTHER.values())

def describe(dfs, keep=None):
    dfs = [df for df in dfs if keep is None or df['EncounterKey'].iloc[0] in keep]
    if not dfs:
        return {'Number of encounters': '0'}
    big = pd.concat([df[['EncounterKey'] + [c for c in num_cols if c in df.columns]] for df in dfs], ignore_index=True)
    per_enc = big.groupby('EncounterKey').mean()                       # encounter mean of 8-hour means
    ks = per_enc.index
    pk = pd.Series([enc_pat[k] for k in ks], index=ks)
    p = pat.reindex(pk.astype('int64').values); p.index = ks
    e = enc.reindex(ks.astype('int64')); e.index = ks
    n = len(ks)
    out = {'Number of patients': f"{pk.nunique():,}", 'Number of encounters': f"{n:,}"}
    if 'BirthDate' in p and p['BirthDate'].notna().any():
        age = (e['AdmissionDate'] - pd.to_datetime(p['BirthDate'], errors='coerce')).dt.days / 365.25
    else:
        age = p['AgeInYears']
    out['Age at admission, mean (SD)'] = f"{age.mean():.1f} ({age.std():.1f})"
    cnt = lambda s, v: f"{(s == v).sum():,} ({100 * (s == v).mean():.1f}%)"
    sex = p['Sex'].astype(str)
    out['Men, n (%)'] = cnt(sex, 'Male'); out['Women, n (%)'] = cnt(sex, 'Female')
    out['Sex unknown, n (%)'] = f"{(~sex.isin(['Male', 'Female'])).sum():,} ({100 * (~sex.isin(['Male', 'Female'])).mean():.1f}%)"
    for lab, fn, col, cats in [('Race', race, 'FirstRace', ['White or Caucasian', 'Other', 'Black or African American', 'Unknown']),
                               ('Ethnicity', ethnicity, 'Ethnicity', ['Not Hispanic or Latino', 'Hispanic or Latino', 'Unknown']),
                               ('Smoking', smoking, 'SmokingStatus', ['Never smoker', 'Former smoker', 'Current smoker', 'Never assessed', 'Other / unknown'])]:
        s = p[col].map(fn)
        for c in cats:
            out[f'{lab}: {c}'] = cnt(s, c)
    s = e['AdmissionType'].map(admission)
    for c in ['Emergency', 'Urgent', 'Elective', 'Other']:
        out[f'Admission: {c}'] = cnt(s, c)
    los = pd.cut(e['LOS'], [-1, 2, 5, 8, np.inf], labels=['0-2', '3-5', '6-8', '9+'])
    for c in ['0-2', '3-5', '6-8', '9+']:
        out[f'Length of stay {c} days'] = cnt(los.astype(str), c)
    for lab, col in {**VITALS, **LABS, **OTHER}.items():
        if col not in per_enc: continue
        v = per_enc[col].astype(float)
        if col == 'Mean Height' and v.median() < 100: v = v * 2.54            # inches -> cm
        if col == 'Mean Weight' and v.median() > 500: v = v * 0.0283495231   # ounces -> kg
        out[lab + ', mean (SD)'] = f"{v.mean():.1f} ({v.std():.1f})"
    return out

table1 = pd.DataFrame({g: describe(dfs) for g, dfs in pools.items()})
table1.to_csv('table1.csv')
if 'epic_encs' in globals():
    table1_epic = pd.DataFrame({g: describe(dfs, keep=epic_encs) for g, dfs in pools.items()})
    table1_epic.to_csv('table1_epic_population.csv')
print("Raw categories, for checking the groupings:")
print(pat['FirstRace'].value_counts().head(12).to_dict()); print(pat['Ethnicity'].value_counts().to_dict())
print(pat['SmokingStatus'].value_counts().to_dict()); print(enc['AdmissionType'].value_counts().to_dict())
with pd.option_context('display.max_rows', None, 'display.width', 250):
    display(table1)

In [ ]:
der = {df['EncounterKey'].iloc[0] for df in filtered_data}
allk = {df['EncounterKey'].iloc[0] for df in filtered_patients_data_cleaned + filtered_patients_data_non_cleaned}
removed = allk - der
m = len(removed & set(non_septic_encs))
sep_removed = len(removed & {df['EncounterKey'].iloc[0] for df in filtered_patients_data})
non_removed = len(removed) - sep_removed - m

print("Septic after exclusions            :", 66105)
print("Derivation cohort (septic)         :", 65756 - sep_removed - m)
print("Validation cohort                  :", 349)
print("Reclassified by Llama              :", 8401 - m)
print("Final septic, derivation           :", 65756 - sep_removed - m - (8401 - m))
print("Non-septic after exclusions        :", 286310 - non_removed)
print("Final non-septic, derivation       :", 286310 - non_removed + 8401 - m)
print("  SIRS+ / SIRS-                    :", 150807, "/", 143898)
print("check: removed", len(removed), "=", sep_removed, "septic +", m, "Llama-reclassified +", non_removed, "non-septic")

In [ ]:
grp = {g: {int(df['EncounterKey'].iloc[0]) for df in pools[g]}
       for g in ('Septic (derivation)', 'Septic (adjudicated)', 'Non-septic (adjudicated)')}
codes = _sql_chunks("SELECT DISTINCT e.EncounterKey, t.Value AS Code, t.DisplayString "
                    "FROM CAB_EPIC.dbo.DiagnosisEventFact e "
                    "JOIN CAB_EPIC.dbo.DiagnosisDim d ON d.DiagnosisKey = e.DiagnosisKey "
                    "JOIN CAB_EPIC.dbo.DiagnosisTerminologyDim t ON t.DiagnosisKey = e.DiagnosisKey AND t.Type = 'ICD-10-CM' "
                    "WHERE d.Name LIKE '%sepsis%' AND e.EncounterKey IN ({})", set().union(*grp.values()))
codes['EncounterKey'] = codes['EncounterKey'].astype('int64')
desc = codes.groupby('Code')['DisplayString'].first()
out = {}
for g, k in grp.items():
    c = codes[codes['EncounterKey'].isin(k)]
    print(f"{g}: {c['EncounterKey'].nunique():,} of {len(k):,} encounters have at least one mapped ICD-10-CM code")
    out[g] = c.groupby('Code')['EncounterKey'].nunique().map(lambda n, N=len(k): f"{n:,} ({100 * n / N:.1f}%)")
icd_tab = pd.DataFrame(out).fillna('')
icd_tab.insert(0, 'Description', desc)
icd_tab = icd_tab.loc[codes[codes['EncounterKey'].isin(grp['Septic (derivation)'])].groupby('Code')['EncounterKey'].nunique().sort_values(ascending=False).index.union(icd_tab.index, sort=False)]
icd_tab.to_csv('paper_icd10_codes.csv'); display(icd_tab.head(25))

In [ ]:
# ===== RANDOM FOREST vs XGBOOST (same splits, imputation, scaling and labels) =====
from sklearn.ensemble import RandomForestClassifier
RF_PARAMS = dict(n_estimators=100, max_depth=12, n_jobs=64, random_state=0)   # same size/depth as XGBoost
RF_GROUPS = {'Septic subset': septic_encounters, 'SIRS+ comparison': sirs_pos_encounters,
             'SIRS- comparison': sirs_neg_encounters}
res = defaultdict(lambda: {'point': [], 'boot': []})
for h, w in WINDOW_SIZES.items():
    splits = build_bootstrapped_splits(w)                       # cached: no new imputation
    for i, s in enumerate(splits):
        t0 = time.time()
        rf = RandomForestClassifier(**RF_PARAMS).fit(s['X_train'], s['y_train'])
        for c in ('test', 'val'):
            y = np.asarray(s['y_' + c]).astype(int); enc = np.asarray(s[c + '_encounter_keys'])
            assert np.array_equal(y, np.asarray(all_curve_data[h][c]['model']['y_true'][i]))
            scores = {'XGBoost': np.asarray(all_curve_data[h][c]['model']['y_score'][i], float),
                      'Random Forest': rf.predict_proba(s['X_' + c])[:, 1]}
            masks = {'Overall': np.ones(len(y), bool)}
            if c == 'test':
                masks.update({g: np.isin(enc, list(k)) for g, k in RF_GROUPS.items()})
            rks = {(m, g): Ranked(y[mk], p[mk]) for m, p in scores.items() for g, mk in masks.items()}
            for (m, g), rk in rks.items():
                res[(h, c, m, g)]['point'].append(rk.auc(np.ones(masks[g].sum())))
            _, inv = np.unique([enc_to_pat[e] for e in enc], return_inverse=True); n = inv.max() + 1
            rng = np.random.default_rng(2000 * h + i)
            for _ in range(B_PER_SPLIT):                        # same patient resample for both models
                wb = rng.multinomial(n, np.full(n, 1 / n)).astype(float)[inv]
                for (m, g), rk in rks.items():
                    res[(h, c, m, g)]['boot'].append(rk.auc(wb[masks[g]]))
        del rf
        print(f"{h}h split {i + 1}/5 done in {(time.time() - t0) / 60:.1f} min")
    del splits
rows = []
for (h, c, m, g), v in res.items():
    lo, hi = np.percentile(v['boot'], [2.5, 97.5])
    rows.append({'Window': f'{h}h', 'Cohort': 'Internal test' if c == 'test' else 'Adjudicated',
                 'Model': m, 'Group': g, 'ROC-AUC': f"{np.mean(v['point']):.3f} ({lo:.3f}-{hi:.3f})"})
rf_table = pd.DataFrame(rows); rf_table.to_csv('paper_rf_vs_xgb.csv', index=False); display(rf_table)

In [ ]:
display(rf_table)